# Unified Quarterly Intelligence Flow

One orchestrated notebook for a company-quarter event. It discovers or loads financial results, investor presentations, and earnings-call transcripts, then routes each document to the right catalog-backed extraction branch before merging facts, metrics, signals, and cards.

## STEP 1 / 7 - COMPANY


In [ ]:
# =============================================================================
# STEP 1 / 7  -  COMPANY
# Register the company and configure the unified document queue.
# =============================================================================
import hashlib
import json
import sqlite3
from pathlib import Path

# --- Config: change these for any NSE-listed symbol/event/document mix ---
SYMBOL = "AXISBANK"
FROM_DATE = "01-10-2024"
TO_DATE = "30-12-2024"
EVENT_TYPE = "quarterly_result"

# IR-agent source discovery config. Required only when any document uses source_mode="ir_agent".
IR_AGENT_COMPANY_URL = "https://www.thermaxglobal.com/investor-overview"  # Example: "https://www.company.com/investors"
IR_AGENT_COMPANY_NAME = SYMBOL
IR_AGENT_MODEL = "gpt-5.5"

# Unified document queue. Each document is routed to its own catalog and
# extractor branch in later steps. source_mode options:
#   nse_auto     - discover from NSE corporate announcements
#   ir_agent     - find official IR assets from IR_AGENT_COMPANY_URL/company_url
#   manual_url   - ingest source_url directly
#   local_file   - ingest local_path directly
DOCUMENTS_TO_PROCESS = [
    # {
    #     "document_type": "financial_result",
    #     "source_mode": "nse_auto",
    #     "catalog": "facts.json",
    # },
    {
        "document_type": "investor_presentation",
        "source_mode": "nse_auto",
        "catalog": "investor_presentation_facts.json",
    },
    # {
    #     "document_type": "earnings_call_transcript",
    #     "source_mode": "nse_auto",
    #     "catalog": "earnings_call_facts.json",
    # },
]

# --- Paths (resolved relative to this notebook/repo) ---
NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name in {"7_step_flow", "8_step_flow"} else NOTEBOOK_DIR
FLOW_DIR = REPO_ROOT / "8_step_flow"
DB_PATH = REPO_ROOT / "v4" / "data" / "capital_nerve.db"
CATALOG_DIR = FLOW_DIR / "catalog_mvp"
DOCUMENTS_DIR = REPO_ROOT / "v4" / "data" / "documents"
ENV_PATH = FLOW_DIR / ".env"
CATALOG_PATHS = {
    "financial_result": CATALOG_DIR / "facts.json",
    "investor_presentation": CATALOG_DIR / "investor_presentation_facts.json",
    "earnings_call_transcript": CATALOG_DIR / "earnings_call_facts.json",
}

DB_PATH.parent.mkdir(parents=True, exist_ok=True)
DOCUMENTS_DIR.mkdir(parents=True, exist_ok=True)

# Schema bootstrap (mirrors v3/db.py so the notebook works on a fresh DB).
SCHEMA = """
CREATE TABLE IF NOT EXISTS companies (
    id TEXT PRIMARY KEY, name TEXT NOT NULL, ticker TEXT, exchange TEXT,
    sector TEXT, industry TEXT, isin TEXT UNIQUE
);
CREATE TABLE IF NOT EXISTS events (
    id TEXT PRIMARY KEY, company_id TEXT NOT NULL REFERENCES companies(id),
    event_type TEXT NOT NULL, event_date TEXT NOT NULL, fiscal_year INTEGER,
    fiscal_quarter INTEGER, title TEXT, source_url TEXT, document_id TEXT,
    status TEXT DEFAULT 'processed'
);
CREATE TABLE IF NOT EXISTS documents (
    id TEXT PRIMARY KEY, company_id TEXT NOT NULL REFERENCES companies(id),
    source_url TEXT, storage_path TEXT NOT NULL, sha256 TEXT NOT NULL UNIQUE,
    title TEXT, document_kind TEXT, file_size INTEGER,
    status TEXT DEFAULT 'pending', error_message TEXT,
    ingested_at TEXT DEFAULT (datetime('now'))
);
CREATE TABLE IF NOT EXISTS fact_definitions (
    fact_code TEXT PRIMARY KEY,
    fact_name TEXT NOT NULL,
    fact_category TEXT NOT NULL,
    value_type TEXT NOT NULL,
    standard_unit TEXT,
    preferred_source TEXT
);
CREATE TABLE IF NOT EXISTS fact_observations (
    observation_id TEXT PRIMARY KEY,
    company_id TEXT NOT NULL REFERENCES companies(id),
    event_id TEXT NOT NULL REFERENCES events(id),
    document_id TEXT REFERENCES documents(id),
    fact_code TEXT NOT NULL REFERENCES fact_definitions(fact_code),
    value REAL,
    value_text TEXT,
    unit TEXT,
    period TEXT,
    source_page INTEGER,
    source_text TEXT,
    segment TEXT,
    geography TEXT,
    product TEXT,
    channel TEXT,
    project TEXT,
    customer_type TEXT,
    metric_context TEXT,
    scope_level TEXT,
    scope_name TEXT,
    fact_type TEXT,
    extraction_method TEXT,
    value_lower REAL,
    value_upper REAL,
    sentiment TEXT,
    is_explicit_guidance INTEGER,
    confidence REAL
);
CREATE TABLE IF NOT EXISTS resolved_facts (
    resolved_fact_id TEXT PRIMARY KEY,
    company_id TEXT NOT NULL REFERENCES companies(id),
    event_id TEXT NOT NULL REFERENCES events(id),
    fact_code TEXT NOT NULL REFERENCES fact_definitions(fact_code),
    resolved_value REAL,
    resolved_value_text TEXT,
    unit TEXT,
    segment TEXT,
    geography TEXT,
    product TEXT,
    channel TEXT,
    project TEXT,
    customer_type TEXT,
    metric_context TEXT,
    scope_level TEXT,
    scope_name TEXT,
    fact_type TEXT,
    value_lower REAL,
    value_upper REAL,
    sentiment TEXT,
    is_explicit_guidance INTEGER,
    selected_observation_id TEXT REFERENCES fact_observations(observation_id),
    resolution_status TEXT,
    confidence REAL
);
CREATE TABLE IF NOT EXISTS extracted_values (
    id TEXT PRIMARY KEY, company_id TEXT NOT NULL REFERENCES companies(id),
    event_id TEXT NOT NULL REFERENCES events(id), value_code TEXT NOT NULL,
    value_numeric REAL, value_text TEXT, unit TEXT, period_type TEXT,
    period_start TEXT, period_end TEXT, basis TEXT DEFAULT 'consolidated',
    segment TEXT, geography TEXT, product TEXT, channel TEXT, project TEXT,
    customer_type TEXT, metric_context TEXT, scope_level TEXT, scope_name TEXT,
    fact_type TEXT, value_lower REAL, value_upper REAL, sentiment TEXT,
    is_explicit_guidance INTEGER, source_text TEXT, source_page INTEGER,
    confidence REAL
);
CREATE TABLE IF NOT EXISTS metrics (
    metric_id TEXT PRIMARY KEY,
    company_id TEXT NOT NULL REFERENCES companies(id),
    event_id TEXT NOT NULL REFERENCES events(id),
    metric_code TEXT NOT NULL,
    value REAL,
    unit TEXT,
    input_fact_ids TEXT,
    formula TEXT
);
CREATE TABLE IF NOT EXISTS signals (
    signal_id TEXT PRIMARY KEY,
    company_id TEXT NOT NULL REFERENCES companies(id),
    event_id TEXT NOT NULL REFERENCES events(id),
    signal_code TEXT NOT NULL,
    severity TEXT,
    direction TEXT,
    supporting_metric_ids TEXT,
    supporting_fact_ids TEXT
);
CREATE TABLE IF NOT EXISTS intelligence_cards (
    card_id TEXT PRIMARY KEY,
    company_id TEXT NOT NULL REFERENCES companies(id),
    event_id TEXT NOT NULL REFERENCES events(id),
    card_title TEXT NOT NULL,
    signal_id TEXT NOT NULL REFERENCES signals(signal_id),
    confidence TEXT,
    display_status TEXT DEFAULT 'published'
);
CREATE TABLE IF NOT EXISTS presentation_document_inventory (
    id TEXT PRIMARY KEY,
    company_id TEXT NOT NULL REFERENCES companies(id),
    event_id TEXT NOT NULL REFERENCES events(id),
    document_id TEXT NOT NULL REFERENCES documents(id),
    period_label TEXT,
    inventory_json TEXT NOT NULL,
    extraction_plan_json TEXT NOT NULL,
    created_at TEXT DEFAULT (datetime('now'))
);
CREATE TABLE IF NOT EXISTS presentation_segments (
    id TEXT PRIMARY KEY,
    company_id TEXT NOT NULL REFERENCES companies(id),
    event_id TEXT NOT NULL REFERENCES events(id),
    document_id TEXT NOT NULL REFERENCES documents(id),
    segment_name TEXT NOT NULL,
    segment_slug TEXT,
    aliases_json TEXT,
    slides_json TEXT,
    confidence REAL
);
"""


def db_connect() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def ensure_column(conn: sqlite3.Connection, table: str, column: str, ddl: str) -> None:
    cols = [r["name"] for r in conn.execute(f"PRAGMA table_info({table})").fetchall()]
    if column not in cols:
        conn.execute(f"ALTER TABLE {table} ADD COLUMN {column} {ddl}")


def migrate_metrics_schema(conn: sqlite3.Connection) -> None:
    # Additive only: never drop historical metrics from a notebook run.
    ensure_column(conn, "metrics", "input_fact_ids", "TEXT")
    ensure_column(conn, "metrics", "formula", "TEXT")


def migrate_signals_schema(conn: sqlite3.Connection) -> None:
    # Additive only: keep existing signal rows and add current notebook fields.
    ensure_column(conn, "signals", "supporting_metric_ids", "TEXT")
    ensure_column(conn, "signals", "supporting_fact_ids", "TEXT")


def migrate_fact_value_schema(conn: sqlite3.Connection) -> None:
    ensure_column(conn, "fact_observations", "value_text", "TEXT")
    for column in ("segment", "geography", "product", "channel", "project", "customer_type", "metric_context"):
        ensure_column(conn, "fact_observations", column, "TEXT")
    for column in ("scope_level", "scope_name", "fact_type", "extraction_method", "sentiment"):
        ensure_column(conn, "fact_observations", column, "TEXT")
    for column in ("value_lower", "value_upper"):
        ensure_column(conn, "fact_observations", column, "REAL")
    ensure_column(conn, "fact_observations", "is_explicit_guidance", "INTEGER")

    ensure_column(conn, "resolved_facts", "resolved_value_text", "TEXT")
    for column in ("segment", "geography", "product", "channel", "project", "customer_type", "metric_context"):
        ensure_column(conn, "resolved_facts", column, "TEXT")
    for column in ("scope_level", "scope_name", "fact_type", "sentiment"):
        ensure_column(conn, "resolved_facts", column, "TEXT")
    for column in ("value_lower", "value_upper"):
        ensure_column(conn, "resolved_facts", column, "REAL")
    ensure_column(conn, "resolved_facts", "is_explicit_guidance", "INTEGER")

    ensure_column(conn, "extracted_values", "value_text", "TEXT")
    for column in ("segment", "geography", "product", "channel", "project", "customer_type", "metric_context"):
        ensure_column(conn, "extracted_values", column, "TEXT")
    for column in ("scope_level", "scope_name", "fact_type", "sentiment"):
        ensure_column(conn, "extracted_values", column, "TEXT")
    for column in ("value_lower", "value_upper"):
        ensure_column(conn, "extracted_values", column, "REAL")
    ensure_column(conn, "extracted_values", "is_explicit_guidance", "INTEGER")

def normalize_document_type(value: str) -> str:
    key = str(value or "").strip().lower().replace("-", "_").replace(" ", "_")
    aliases = {
        "financial_results": "financial_result",
        "result": "financial_result",
        "results": "financial_result",
        "presentation": "investor_presentation",
        "investor_presentation_pdf": "investor_presentation",
        "earnings_call": "earnings_call_transcript",
        "earnings_transcript": "earnings_call_transcript",
        "transcript": "earnings_call_transcript",
        "concall_transcript": "earnings_call_transcript",
    }
    return aliases.get(key, key)


def catalog_path_for(document_type: str, configured: str | None = None) -> Path:
    if configured:
        p = Path(configured)
        if not p.is_absolute():
            p = CATALOG_DIR / p
        if p.exists():
            return p
    return CATALOG_PATHS[normalize_document_type(document_type)]


def seed_fact_definitions(conn: sqlite3.Connection) -> None:
    requested_catalogs = []
    for doc in DOCUMENTS_TO_PROCESS:
        doc_type = normalize_document_type(doc.get("document_type"))
        path = catalog_path_for(doc_type, doc.get("catalog"))
        requested_catalogs.append((doc_type, path))
    seen: set[str] = set()
    for doc_type, facts_path in requested_catalogs:
        if str(facts_path) in seen:
            continue
        seen.add(str(facts_path))
        if not facts_path.exists():
            print(f"Skipping missing fact catalog: {facts_path}")
            continue
        catalog = json.loads(facts_path.read_text(encoding="utf-8"))
        for code, spec in catalog.items():
            docs = spec.get("documents") or []
            preferred_source = normalize_document_type(docs[0]) if docs else doc_type
            unit = spec.get("unit")
            standard_unit = "INR crore" if unit == "crore" else unit
            fact_type = str(spec.get("fact_type") or "").lower()
            value_type = "text" if fact_type in {"text", "categorical", "date"} or unit in {"text", "date", "enum"} else "numeric"
            conn.execute(
                """
                INSERT INTO fact_definitions (
                    fact_code, fact_name, fact_category, value_type,
                    standard_unit, preferred_source
                ) VALUES (?, ?, ?, ?, ?, ?)
                ON CONFLICT(fact_code) DO UPDATE SET
                    fact_name = excluded.fact_name,
                    fact_category = excluded.fact_category,
                    value_type = excluded.value_type,
                    standard_unit = excluded.standard_unit,
                    preferred_source = excluded.preferred_source
                """,
                (
                    code,
                    spec.get("name", code),
                    str(spec.get("statement") or "financial").lower(),
                    value_type,
                    standard_unit,
                    preferred_source,
                ),
            )


# Consistent company id derived from the symbol (kept the same in every step).
company_id = hashlib.sha256(f"{SYMBOL}:NSE".encode()).hexdigest()

with db_connect() as conn:
    conn.executescript(SCHEMA)
    migrate_metrics_schema(conn)
    migrate_signals_schema(conn)
    migrate_fact_value_schema(conn)
    seed_fact_definitions(conn)
    conn.execute(
        """
        INSERT INTO companies (id, name, ticker, exchange)
        VALUES (?, ?, ?, 'NSE')
        ON CONFLICT(id) DO UPDATE SET ticker = excluded.ticker
        """,
        (company_id, SYMBOL, SYMBOL),
    )
    conn.commit()
    row = conn.execute("SELECT * FROM companies WHERE id = ?", (company_id,)).fetchone()

print(f"DB: {DB_PATH}")
print(f"Registered company {SYMBOL}  (company_id={company_id[:12]}...)")
print(dict(row))

## STEP 2 / 7  -  EVENT

In [ ]:
# =============================================================================
# STEP 2 / 7  -  EVENT
# Optionally call NSE for auto-discovery, then persist discovered announcements.
# Local/manual document workflows can skip NSE entirely.
# =============================================================================
import requests

_API_URL = "https://www.nseindia.com/api/NextApi/apiClient/GetQuoteApi"
_HOMEPAGE = "https://www.nseindia.com"
_PAGE_URL = "https://www.nseindia.com/companies-listing/corporate-filings-announcements"

# NSE blocks generic Python user agents without a session cookie.
_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": _PAGE_URL,
}

params = {
    "functionName": "getCorporateAnnouncement",
    "symbol": SYMBOL,
    "marketApiType": "equities",
    "subject": "",
    "fromDate": FROM_DATE,
    "toDate": TO_DATE,
}

# `session` is reused in Step 4 to download documents.
session = requests.Session()
session.headers.update(_HEADERS)

NEEDS_NSE_DISCOVERY = any(
    str(doc.get("source_mode", "nse_auto")).strip().lower() == "nse_auto"
    for doc in DOCUMENTS_TO_PROCESS
)

if NEEDS_NSE_DISCOVERY:
    session.get(_HOMEPAGE, timeout=30)  # warm up cookies
    response = session.get(_API_URL, params=params, timeout=30)
    response.raise_for_status()
    announcements = response.json()
else:
    announcements = []


def event_id_from_source(source_url: str) -> str:
    return hashlib.sha256(f"{company_id}:{source_url}".encode()).hexdigest()


# Backfill company name / ISIN from the first announcement (same company_id).
# The ISIN column is UNIQUE; an earlier (ISIN-keyed) run may already own it, so
# only claim the ISIN when it is free or already ours.
if announcements:
    first = announcements[0]
    isin = first.get("sm_isin")
    with db_connect() as conn:
        if isin:
            owner = conn.execute(
                "SELECT id FROM companies WHERE isin = ?", (isin,)
            ).fetchone()
            if owner is not None and owner["id"] != company_id:
                isin = None  # taken by a different company row -> don't reclaim
        conn.execute(
            """
            UPDATE companies
            SET name = COALESCE(?, name), isin = COALESCE(?, isin),
                industry = COALESCE(?, industry)
            WHERE id = ?
            """,
            (
                first.get("sm_name") or SYMBOL,
                isin,
                first.get("smIndustry"),
                company_id,
            ),
        )
        conn.commit()

# Persist every discovered NSE announcement as an event candidate.
stored = 0
with db_connect() as conn:
    for item in announcements:
        source_url = item.get("attchmntFile") or ""
        seed = source_url or f"{item.get('desc')}:{item.get('dt')}"
        ev_id = event_id_from_source(seed)
        conn.execute(
            """
            INSERT OR IGNORE INTO events (
                id, company_id, event_type, event_date, title, source_url, status
            ) VALUES (?, ?, ?, ?, ?, ?, 'discovered')
            """,
            (
                ev_id,
                company_id,
                (item.get("desc") or "(missing)").strip(),
                (item.get("sort_date") or "")[:10],
                item.get("attchmntText"),
                source_url or None,
            ),
        )
        stored += 1
    conn.commit()

by_desc: dict[str, list[dict]] = {}
for item in announcements:
    desc = (item.get("desc") or "(missing)").strip()
    by_desc.setdefault(desc, []).append(item)

if NEEDS_NSE_DISCOVERY:
    print(f"{len(announcements)} announcements ({stored} stored as events) "
          f"across {len(by_desc)} desc buckets for {SYMBOL} [{FROM_DATE} -> {TO_DATE}]\n")
    for desc, items in sorted(by_desc.items(), key=lambda kv: (-len(kv[1]), kv[0])):
        print(f"  {len(items):>3}  {desc}")
else:
    print("NSE discovery skipped; using Step 1 local/manual/IR-agent document configuration.")



## STEP 3 / 7 - EVENT TYPE

Classify exchange announcements and build one ingestion queue for financial results, investor presentations, and earnings-call transcripts.


In [ ]:
# =============================================================================
# STEP 3 / 7  -  EVENT TYPE
# Classify announcements into known document buckets and build a generic
# document-ingestion queue from Step 1's DOCUMENTS_TO_PROCESS.
# =============================================================================
import sys

V3_ROOT = (REPO_ROOT / "v3").resolve()
if str(V3_ROOT) not in sys.path:
    sys.path.insert(0, str(V3_ROOT))

from nse_fr_resolver import infer_period_markers, resolve_canonical_financial_report


def _text_blob(item: dict) -> str:
    parts = [
        item.get("desc") or "",
        item.get("attchmntText") or "",
        item.get("attchmntFile") or "",
    ]
    return " ".join(parts).lower()


def classify_announcement(item: dict) -> str:
    """Trimmed classifier covering the three event-type buckets this flow knows."""
    desc = (item.get("desc") or "").strip()
    blob = _text_blob(item)

    def has(*phrases: str) -> bool:
        return any(p in blob for p in phrases)

    # Earnings call transcript. NSE titles are often terse, e.g. just "Transcript".
    transcript_update = desc == "Analysts/Institutional Investor Meet/Con. Call Updates" and has("transcript")
    non_transcript_meet_update = has("schedule of meet", "audio recording", "link of recording")
    if has("transcript of the discussion", "earnings call transcript", "concall transcript") or (transcript_update and not non_transcript_meet_update):
        return "Earnings Call Transcript"

    # Investor presentation.
    if desc == "Investor Presentation" or has("investor presentation"):
        return "Investor Presentation"

    # Financial results: the actual results filing, not an intimation/conf-call.
    intimation = has(
        "scheduled to be held", "audio recording", "will hold", "will be held", "informed the exchange regarding board meeting",
        "conference call", "prior intimation", "recording and transcript", "media release", "shareholder meeting"
    )
    if desc == "Financial Results" and not intimation:
        return "Financial Results"
    if has("financial results", "unaudited financial", "audited financial", "outcome of board meeting") and not intimation:
        return "Financial Results"

    return "Other"


# Attach classification + parsed timestamp to each announcement.
from datetime import datetime

for item in announcements:
    item["event_bucket"] = classify_announcement(item)

by_bucket: dict[str, list[dict]] = {}
for item in announcements:
    by_bucket.setdefault(item["event_bucket"], []).append(item)

financial_results = by_bucket.get("Financial Results", [])


def _sort_key(item: dict):
    try:
        return datetime.strptime(item.get("sort_date", ""), "%Y-%m-%d %H:%M:%S")
    except ValueError:
        return datetime.min


for items in by_bucket.values():
    items.sort(key=_sort_key, reverse=True)

period_markers = infer_period_markers(announcements)
documents_to_ingest: list[dict] = []
resolved_fr = None
chosen = None
chosen_source_url = None

BUCKET_BY_DOCUMENT_TYPE = {
    "financial_result": "Financial Results",
    "investor_presentation": "Investor Presentation",
    "earnings_call_transcript": "Earnings Call Transcript",
}

IR_AGENT_ASSET_BY_DOCUMENT_TYPE = {
    "financial_result": "financial_result",
    "investor_presentation": "investor_presentation",
    "earnings_call_transcript": "earnings_transcript",
}



def _event_seed_for_doc(doc: dict) -> str:
    return doc.get("source_url") or doc.get("local_path") or doc.get("document_type") or EVENT_TYPE


def _to_ir_agent_date(value: str) -> str:
    for fmt in ("%d-%m-%Y", "%Y-%m-%d"):
        try:
            return datetime.strptime(value, fmt).date().isoformat()
        except ValueError:
            pass
    raise ValueError(f"IR agent date must be DD-MM-YYYY or YYYY-MM-DD, got {value!r}")


def _ir_agent_config(request: dict) -> tuple[str, str | None, str, str, str | None]:
    company_url = request.get("company_url") or IR_AGENT_COMPANY_URL
    if not company_url:
        raise ValueError(
            f"source_mode='ir_agent' for {request.get('document_type')} requires "
            "IR_AGENT_COMPANY_URL or document-level company_url"
        )
    company_name = request.get("company_name") or IR_AGENT_COMPANY_NAME or SYMBOL
    start_date = request.get("start_date") or _to_ir_agent_date(FROM_DATE)
    end_date = request.get("end_date") or _to_ir_agent_date(TO_DATE)
    model = request.get("model") or IR_AGENT_MODEL or None
    return company_url, company_name, start_date, end_date, model


def _load_ir_agent_finder():
    try:
        from dotenv import load_dotenv

        if ENV_PATH.exists():
            load_dotenv(ENV_PATH)
    except ImportError:
        pass

    ir_agent_root = (REPO_ROOT / "IR_agent").resolve()
    if str(ir_agent_root) not in sys.path:
        sys.path.insert(0, str(ir_agent_root))

    from ir_agent import find_ir_assets

    return find_ir_assets


find_ir_assets = _load_ir_agent_finder() if any(
    str(doc.get("source_mode", "nse_auto")).strip().lower() == "ir_agent" for doc in DOCUMENTS_TO_PROCESS
) else None
ir_agent_result_cache: dict[tuple[str, str | None, str, str, str | None], object] = {}


def _resolve_ir_agent_match(request: dict, doc_type: str):
    if find_ir_assets is None:
        raise RuntimeError("IR agent source requested but finder was not loaded")
    config = _ir_agent_config(request)
    if config not in ir_agent_result_cache:
        company_url, company_name, start_date, end_date, model = config
        print(f"Searching IR assets for {company_name} [{start_date} -> {end_date}]...")
        ir_agent_result_cache[config] = find_ir_assets(
            company_url=company_url,
            company_name=company_name,
            start_date=start_date,
            end_date=end_date,
            model=model,
        )
    result = ir_agent_result_cache[config]
    requested_asset_type = IR_AGENT_ASSET_BY_DOCUMENT_TYPE.get(doc_type)
    min_confidence = float(request.get("min_confidence", 0.0))
    matches = [
        match for match in result.matches
        if match.asset_type == requested_asset_type and match.confidence >= min_confidence
    ]
    matches.sort(
        key=lambda match: (match.confidence, match.published_or_period_date or ""),
        reverse=True,
    )
    return matches[0] if matches else None


for request in DOCUMENTS_TO_PROCESS:
    doc_type = normalize_document_type(request.get("document_type"))
    source_mode = str(request.get("source_mode", "nse_auto")).strip().lower()
    catalog_path = catalog_path_for(doc_type, request.get("catalog"))

    if source_mode == "local_file":
        local_path = request.get("local_path")
        if not local_path:
            print(f"Skipping {doc_type}: local_file requested but local_path is empty")
            continue
        documents_to_ingest.append({
            "document_type": doc_type,
            "source_mode": source_mode,
            "local_path": str(Path(local_path).expanduser()),
            "title": request.get("title") or f"{SYMBOL} {doc_type}",
            "catalog_path": catalog_path,
            "announcement": None,
        })
        continue

    if source_mode == "manual_url":
        source_url = request.get("source_url")
        if not source_url:
            print(f"Skipping {doc_type}: manual_url requested but source_url is empty")
            continue
        documents_to_ingest.append({
            "document_type": doc_type,
            "source_mode": source_mode,
            "source_url": source_url,
            "title": request.get("title") or source_url.rsplit("/", 1)[-1],
            "catalog_path": catalog_path,
            "announcement": None,
        })
        continue

    if source_mode == "ir_agent":
        selected = _resolve_ir_agent_match(request, doc_type)
        if selected is None:
            print(f"No IR-agent match found for requested {doc_type}")
            continue
        sort_date = f"{selected.published_or_period_date} 00:00:00" if selected.published_or_period_date else None
        documents_to_ingest.append({
            "document_type": doc_type,
            "source_mode": source_mode,
            "source_url": selected.url,
            "title": request.get("title") or selected.title,
            "sort_date": sort_date,
            "catalog_path": catalog_path,
            "announcement": None,
            "source_page": selected.source_page,
            "ir_agent_confidence": selected.confidence,
            "ir_agent_period_label": selected.period_label,
            "ir_agent_notes": selected.notes,
        })
        continue

    if source_mode != "nse_auto":
        print(f"Skipping {doc_type}: unsupported source_mode={source_mode!r}")
        continue

    if doc_type == "financial_result":
        if not financial_results:
            print(f"No Financial Results announcement found for {SYMBOL} in {FROM_DATE} -> {TO_DATE}")
            continue
        print(f"Resolving canonical financial report ({len(financial_results)} candidate(s), "
              f"{len(period_markers)} period marker(s))...")
        resolved_fr = resolve_canonical_financial_report(
            announcements,
            financial_results,
            period_markers=period_markers or None,
            session=session,
            referer=_PAGE_URL,
        )
        if not resolved_fr:
            print(f"No valid financial results PDF found for {SYMBOL}")
            continue
        chosen = resolved_fr["announcement"]
        chosen_source_url = resolved_fr["url"]
        documents_to_ingest.append({
            "document_type": doc_type,
            "source_mode": source_mode,
            "source_url": chosen_source_url,
            "pdf_bytes": resolved_fr["pdf_bytes"],
            "title": chosen.get("attchmntText"),
            "sort_date": chosen.get("sort_date"),
            "catalog_path": catalog_path,
            "announcement": chosen,
            "classification": resolved_fr.get("classification"),
        })
        continue

    bucket = BUCKET_BY_DOCUMENT_TYPE.get(doc_type)
    candidates = by_bucket.get(bucket, []) if bucket else []
    selected = next((a for a in candidates if a.get("attchmntFile")), None)
    if selected is None:
        print(f"No NSE candidate found for requested {doc_type}; add local_path/source_url in Step 1 when ready")
        continue
    documents_to_ingest.append({
        "document_type": doc_type,
        "source_mode": source_mode,
        "source_url": selected.get("attchmntFile"),
        "title": selected.get("attchmntText"),
        "sort_date": selected.get("sort_date"),
        "catalog_path": catalog_path,
        "announcement": selected,
    })

if not documents_to_ingest:
    raise RuntimeError("No documents resolved. Add at least one valid nse_auto, ir_agent, manual_url, or local_file document in Step 1.")

EVENT_TYPE_BY_DOCUMENT_TYPE = {
    "financial_result": EVENT_TYPE,
    "investor_presentation": "investor_presentation",
    "earnings_call_transcript": "earnings_call_transcript",
}


def _event_type_for_doc(doc: dict) -> str:
    return EVENT_TYPE_BY_DOCUMENT_TYPE.get(normalize_document_type(doc.get("document_type")), EVENT_TYPE)


def _event_date_for_doc(doc: dict) -> str:
    sort_date = doc.get("sort_date") or (doc.get("announcement") or {}).get("sort_date")
    if sort_date:
        return str(sort_date)[:10]
    return datetime.today().date().isoformat()


for doc in documents_to_ingest:
    doc["event_id"] = event_id_from_source(_event_seed_for_doc(doc))

primary_doc = next((d for d in documents_to_ingest if d["document_type"] == "financial_result"), documents_to_ingest[0])
event_id = primary_doc["event_id"]

with db_connect() as conn:
    for doc in documents_to_ingest:
        conn.execute(
            """
            INSERT INTO events (id, company_id, event_type, event_date, title, source_url, status)
            VALUES (?, ?, ?, ?, ?, ?, 'selected')
            ON CONFLICT(id) DO UPDATE SET
                event_type = excluded.event_type,
                event_date = COALESCE(NULLIF(excluded.event_date, ''), events.event_date),
                title = COALESCE(excluded.title, events.title),
                source_url = COALESCE(excluded.source_url, events.source_url),
                status = 'selected'
            """,
            (doc["event_id"], company_id, _event_type_for_doc(doc), _event_date_for_doc(doc),
             doc.get("title"), doc.get("source_url")),
        )
    conn.commit()

print("Documents queued for ingestion:")
for doc in documents_to_ingest:
    print(f"  {doc['document_type']:<28} event={doc['event_id'][:12]}...  {doc.get('source_url') or doc.get('local_path')}")
if resolved_fr:
    cls = resolved_fr.get("classification") or {}
    print("\nChosen Financial Result:")
    print(f"  date : {chosen.get('sort_date')}")
    print(f"  title: {chosen.get('attchmntText')}")
    print(f"  pdf  : {chosen_source_url}")
    print(f"  pdf classification: is_fr={cls.get('is_financial_report')} "
          f"conf={cls.get('confidence')} kind={cls.get('document_kind')}")
print(f"  primary_event_id={event_id[:12]}...")

## STEP 4 / 7 - VALUES

Parse each queued document, load the document-specific catalog from `catalog_mvp`, extract observations, and resolve trusted facts for the current event.


In [ ]:
# =============================================================================
# STEP 4 / 7  -  VALUES
# Ingest every document queued in Step 3, parse to markdown/text, extract typed
# fact observations with the document-specific catalog, and resolve trusted facts.
# =============================================================================
import json
import re
import sys
from datetime import date, datetime

# ---- 4a. Load OPENAI_API_KEY from .env ---------------------------------------
def _load_env_value(key: str) -> str:
    for line in ENV_PATH.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line.startswith(f"{key}="):
            return line.split("=", 1)[1].strip()
    return ""


OPENAI_API_KEY = _load_env_value("OPENAI_API_KEY")
OPENAI_MODEL = _load_env_value("OPENAI_MODEL") or "gpt-4.1-mini"
OPENAI_PARSE_MODEL = _load_env_value("OPENAI_PARSE_MODEL") or OPENAI_MODEL
if not OPENAI_API_KEY:
    raise RuntimeError(f"OPENAI_API_KEY not found in {ENV_PATH}")

PARSED_DIR = (REPO_ROOT / "v4" / "data" / "parsed").resolve()
PARSED_DIR.mkdir(parents=True, exist_ok=True)

# ---- 4b. Parser imports -------------------------------------------------------
V2_ROOT = (REPO_ROOT / "v2").resolve()
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

import importlib

import pdf_parse
import quarter_column
from periods import (
    detect_reporting_period,
    fy_start_year_from_date,
    quarter_end_date,
    quarter_from_date,
    reporting_period_from_date,
)

importlib.reload(pdf_parse)
importlib.reload(quarter_column)

from openai import OpenAI
from pdf_parse import parse_pdf_to_markdown
from quarter_column import (
    extract_facts_from_quarter_column,
    extract_scoped_facts_from_quarter_columns,
)

client = OpenAI(api_key=OPENAI_API_KEY)

# ---- 4c. Generic document loading/parsing ------------------------------------
def _download_bytes(url: str) -> bytes:
    response = session.get(url, timeout=45)
    response.raise_for_status()
    return response.content


def _store_document_bytes(doc: dict, payload: bytes, suffix: str = ".pdf") -> tuple[str, Path]:
    doc_hash = hashlib.sha256(payload).hexdigest()
    storage_path = DOCUMENTS_DIR / f"{doc_hash}{suffix}"
    if not storage_path.exists():
        storage_path.write_bytes(payload)
    return doc_hash, storage_path


def _document_payload(doc: dict) -> tuple[str, Path, bytes | None, str | None]:
    local_path = doc.get("local_path")
    if local_path:
        path = Path(local_path).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(f"Document not found: {path}")
        payload = path.read_bytes()
        doc_hash = hashlib.sha256(payload).hexdigest()
        return doc_hash, path, payload, path.suffix.lower()

    payload = doc.get("pdf_bytes")
    source_url = doc.get("source_url")
    if payload is None and source_url:
        payload = _download_bytes(source_url)
    if payload is None:
        raise RuntimeError(f"No payload/source available for {doc.get('document_type')}")
    suffix = ".pdf" if payload[:4] == b"%PDF" else ".bin"
    doc_hash, storage_path = _store_document_bytes(doc, payload, suffix=suffix)
    return doc_hash, storage_path, payload, suffix


def _parse_document(doc: dict, storage_path: Path, payload: bytes | None, suffix: str | None) -> str:
    doc_type = doc["document_type"]
    is_text = suffix in {".txt", ".md", ".markdown"}
    if is_text:
        return storage_path.read_text(encoding="utf-8", errors="ignore")
    if payload and payload[:4] != b"%PDF" and doc_type == "earnings_call_transcript":
        return payload.decode("utf-8", errors="ignore")
    return parse_pdf_to_markdown(
        storage_path,
        parsed_dir=PARSED_DIR,
        client=client,
        model=OPENAI_PARSE_MODEL,
    )


parsed_documents: list[dict] = []
for doc in documents_to_ingest:
    doc = dict(doc)
    doc["document_type"] = normalize_document_type(doc["document_type"])
    doc_id, storage_path, payload, suffix = _document_payload(doc)
    markdown = _parse_document(doc, storage_path, payload, suffix)
    doc.update({
        "document_id": doc_id,
        "storage_path": storage_path,
        "file_size": len(payload) if payload is not None else storage_path.stat().st_size,
        "markdown": markdown,
    })
    parsed_documents.append(doc)
    with db_connect() as conn:
        conn.execute(
            """
            INSERT OR IGNORE INTO documents (
                id, company_id, source_url, storage_path, sha256, title,
                document_kind, file_size, status
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, 'ingested')
            """,
            (doc_id, company_id, doc.get("source_url"), str(storage_path), doc_id,
             doc.get("title"), doc["document_type"].upper(), doc["file_size"]),
        )
        conn.commit()
    print(f"Parsed {doc['document_type']}: {len(markdown):,} chars")

if not parsed_documents:
    raise RuntimeError("No documents parsed for extraction")

# ---- 4d. Detect reporting period once for the event ---------------------------
period_source = next((d for d in parsed_documents if d["document_type"] == "financial_result"), parsed_documents[0])
reporting_period = detect_reporting_period(
    period_source["markdown"], title=period_source.get("title") or ""
)
if reporting_period is None:
    sort_date = period_source.get("sort_date") or (period_source.get("announcement") or {}).get("sort_date")
    if sort_date:
        ann = datetime.strptime(sort_date, "%Y-%m-%d %H:%M:%S").date()
    else:
        ann = date.today()
    fy = fy_start_year_from_date(ann)
    q = quarter_from_date(ann)
    reporting_period = reporting_period_from_date(
        quarter_end_date(q, fy), "document_fallback"
    )

period_date = date.fromisoformat(reporting_period.quarter_end)
PERIOD_QUARTER = reporting_period.quarter
PERIOD_FY_START = reporting_period.fy_start_year
PERIOD_END = reporting_period.quarter_end
PERIOD_LABEL = reporting_period.label
print(f"Reporting period: {PERIOD_LABEL}  (quarter_end={PERIOD_END})")

# ---- 4e. Catalog-aware extraction --------------------------------------------
def _chunk(text: str, max_chars: int = 12000) -> list[str]:
    if len(text) <= max_chars:
        return [text]
    chunks, start = [], 0
    while start < len(text):
        end = min(len(text), start + max_chars)
        if end < len(text):
            brk = text.rfind("\n\n", start, end)
            if brk > start:
                end = brk
        chunks.append(text[start:end])
        start = end
    return chunks


def split_slides(markdown: str) -> list[dict]:
    matches = list(re.finditer(r"(?m)^# Page\s+(\d+)\s*$", markdown))
    if not matches:
        return [{"slide_no": 1, "text": markdown.strip(), "title": "Document"}]
    slides = []
    for idx, match in enumerate(matches):
        start = match.end()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(markdown)
        slide_no = int(match.group(1))
        text = markdown[start:end].strip("\n- ")
        title = ""
        for line in text.splitlines():
            clean = re.sub(r"^[#*\-\s]+", "", line).strip()
            if clean and not clean.lower().startswith("page ") and clean != "---":
                title = clean[:140]
                break
        slides.append({"slide_no": slide_no, "text": text, "title": title or f"Slide {slide_no}"})
    return slides


def _catalog_index(catalog: dict) -> tuple[dict[str, str], str]:
    storage_to_fact: dict[str, str] = {}
    fact_lines: list[str] = []
    for key, spec in catalog.items():
        storage_to_fact[key] = key
        for alias in spec.get("aliases") or []:
            storage_to_fact[str(alias)] = key
        aliases = ", ".join(spec.get("aliases") or [])
        dims = ", ".join(spec.get("dimensions") or [])
        alias_note = f" aliases={aliases}" if aliases else ""
        dim_note = f" dimensions={dims}" if dims else ""
        fact_type_note = f" fact_type={spec.get('fact_type')}" if spec.get("fact_type") else ""
        fact_lines.append(
            f"- {key}: {spec.get('name')} unit={spec.get('unit')} "
            f"statement={spec.get('statement')}{dim_note}{fact_type_note}{alias_note}"
        )
    return storage_to_fact, "\n".join(fact_lines)


def _fact_value_type(spec: dict) -> str:
    fact_type = str(spec.get("fact_type") or "").lower()
    unit = spec.get("unit")
    return "text" if fact_type in {"text", "categorical", "date"} or unit in {"text", "date", "enum"} else "numeric"


def _json_response(prompt: str) -> dict:
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=[{"role": "user", "content": prompt}],
        text={"format": {"type": "json_object"}},
        temperature=0,
    )
    return json.loads((response.output_text or "{}").strip() or "{}")


def extract_facts_from_chunk(chunk: str, *, doc_type: str, fact_catalog_text: str) -> list[dict]:
    prompt = f"""Extract facts from this Indian corporate disclosure markdown/text.

    Document type: {doc_type}
    Reporting period context: {PERIOD_LABEL}

    Allowed fact_key values (use canonical keys from catalog):
    {fact_catalog_text}

    Rules:
    - Extract ONLY values explicitly present in the document.
    - Do not extract every nearby number; only extract values matching the catalog.
    - Investor presentations are handled by the slide-level extractor; this chunk extractor is for financial results and transcripts.
    - For numeric facts, return numeric_value as a number with commas stripped.
    - For qualitative/text facts, return value_text.
    - basis should be "consolidated", "standalone", or "not_applicable".
    - unit should match the catalog when possible.
    - source_page should be a page number when visible in markdown, else null.
    - evidence/source_text should be a short nearby snippet.
    - confidence: 0.0 to 1.0.
    - If a fact is segment/business specific, include segment as snake_case.
    - If a KPI can occur in multiple contexts, include metric_context as snake_case.
    - If the catalog says a fact has dimensions such as segment, do not return it without that dimension.
    - For CONTEXT/text facts, return concise analyst wording in value_text; do not invent facts.

    Return JSON object: {{"facts": [{{"fact_key": "...", "numeric_value": 0.0, "value_text": "...", "unit": "...", "basis": "consolidated", "segment": null, "geography": null, "metric_context": null, "scope": "CURRENT", "period_end": "YYYY-MM-DD", "period_label": "...", "source_page": 1, "evidence": "...", "confidence": 0.9}}]}}
    If no facts found, return {{"facts": []}}.

    Document text:
    {chunk}
    """
    try:
        payload = _json_response(prompt)
    except Exception as exc:
        print(f"LLM extraction skipped for one chunk: {type(exc).__name__}: {exc}")
        return []
    facts = payload.get("facts") or []
    return [f for f in facts if isinstance(f, dict)]


def _canon_unit(unit):
    if not unit:
        return None
    u = str(unit).strip().lower()
    return {
        "crores": "crore",
        "cr": "crore",
        "rs. cr": "crore",
        "rs cr": "crore",
        "rs. crore": "crore",
        "inr crore": "crore",
        "percent": "%",
        "pct": "%",
    }.get(u, unit)


def _row_unit(spec: dict, entry: dict, value_type: str):
    if value_type == "text":
        return spec.get("unit")
    return _canon_unit(entry.get("unit")) or spec.get("unit")


def _llm_chunk_allowed(chunk: str, doc_type: str) -> bool:
    if doc_type != "financial_result":
        return True
    low = chunk.lower()
    if "standalone unaudited financial results" in low or "statement of standalone" in low:
        return False
    if "segment financial highlights" in low or "specified items of standalone" in low:
        return False
    return True


def _true_restatement_flag(entry: dict) -> bool:
    text = " ".join(str(entry.get(k) or "") for k in ("value_text", "evidence", "source_text")).lower()
    if not text:
        return False
    if any(p in text for p in ("prior period error", "material error", "restated", "restatement", "corrected")):
        return True
    if ("regrouped" in text or "reclassified" in text) and not any(p in text for p in ("restated", "error", "correction")):
        return False
    return False


def _bool_numeric(value) -> float | None:
    if isinstance(value, bool):
        return 1.0 if value else 0.0
    low = str(value or "").strip().lower()
    if low in {"true", "yes", "1"}:
        return 1.0
    if low in {"false", "no", "0"}:
        return 0.0
    return None


def _optional_float(value) -> float | None:
    if value is None or value == "":
        return None
    try:
        return float(str(value).replace(",", ""))
    except (TypeError, ValueError):
        return None


def _optional_bool(value) -> int | None:
    if value is None or value == "":
        return None
    if isinstance(value, bool):
        return 1 if value else 0
    low = str(value).strip().lower()
    if low in {"true", "yes", "1", "explicit"}:
        return 1
    if low in {"false", "no", "0", "implicit"}:
        return 0
    return None


def _dimension_slug(value) -> str | None:
    if value is None:
        return None
    text = re.sub(r"[^a-z0-9]+", "_", str(value).strip().lower()).strip("_")
    return text or None


def _scope_fields(entry: dict) -> dict:
    scope = entry.get("scope") if isinstance(entry.get("scope"), dict) else {}
    level = str(scope.get("level") or entry.get("scope_level") or "unknown").strip().lower()
    name = scope.get("name") or entry.get("scope_name")
    if level == "company":
        name = None
    elif name is not None:
        name = str(name).strip() or None

    fields = {
        "scope_level": level,
        "scope_name": name,
        "segment": _dimension_slug(entry.get("segment") or entry.get("business_segment")),
        "geography": _dimension_slug(entry.get("geography") or entry.get("region")),
        "product": _dimension_slug(entry.get("product")),
        "channel": _dimension_slug(entry.get("channel")),
        "project": _dimension_slug(entry.get("project")),
        "customer_type": _dimension_slug(entry.get("customer_type")),
        "metric_context": _dimension_slug(entry.get("metric_context") or entry.get("context") or entry.get("kpi_context")),
    }
    if level in {"segment", "geography", "product", "channel", "project", "customer_type"} and name:
        fields[level] = _dimension_slug(name)
    return fields


def _generic_dimension_fields(entry: dict) -> dict:
    return {
        "scope_level": entry.get("scope_level"),
        "scope_name": entry.get("scope_name"),
        "segment": _dimension_slug(entry.get("segment") or entry.get("business_segment")),
        "geography": _dimension_slug(entry.get("geography") or entry.get("region")),
        "product": _dimension_slug(entry.get("product")),
        "channel": _dimension_slug(entry.get("channel")),
        "project": _dimension_slug(entry.get("project")),
        "customer_type": _dimension_slug(entry.get("customer_type")),
        "metric_context": _dimension_slug(entry.get("metric_context") or entry.get("context") or entry.get("kpi_context")),
    }


def dimension_key(row: dict) -> tuple[str | None, str | None, str | None, str | None, str | None, str | None, str | None]:
    return (
        row.get("segment"),
        row.get("geography"),
        row.get("product"),
        row.get("channel"),
        row.get("project"),
        row.get("customer_type"),
        row.get("metric_context"),
    )


def _dimension_suffix(row: dict) -> str:
    parts = [p for p in dimension_key(row) if p]
    return f" {{{'/'.join(parts)}}}" if parts else ""


def dimension_sort_key(row: dict) -> tuple[str, str, str, str, str, str, str]:
    return tuple(str(v or "") for v in dimension_key(row))


def build_inventory(doc: dict, slides: list[dict]) -> dict:
    slide_digest = []
    for slide in slides:
        text = re.sub(r"\s+", " ", slide["text"]).strip()
        slide_digest.append({
            "slide_no": slide["slide_no"],
            "title": slide["title"],
            "text_sample": text[:2200],
        })
    prompt = f"""You are mapping an Indian listed-company investor presentation before fact extraction.

Company symbol: {SYMBOL}
Document title: {doc.get('title') or ''}
Reporting period context: {PERIOD_LABEL}

Task:
- Identify presentation sections, business segment names, aliases, and the slides where each segment appears.
- Mark slides as company-level, segment-specific, multi-segment, geography/product/channel/project/customer-type, disclaimer, or noise.
- Focus on financial highlights, segment performance, business KPIs, order book/pipeline, guidance/outlook, and capex/capacity.
- Do not extract final facts in this pass.

Return JSON object:
{{
  "document_type": "investor_presentation",
  "period_label": "...",
  "detected_sections": [{{"name": "...", "slides": [1, 2]}}],
  "detected_segments": [{{"name": "...", "aliases": ["..."], "slides": [3, 4], "confidence": 0.0}}],
  "detected_geographies": [{{"name": "...", "slides": [5]}}],
  "detected_metrics": ["revenue", "order book"],
  "slide_plan": [{{"slide_no": 1, "title": "...", "primary_scope_level": "company|segment|multi_segment|geography|product|channel|project|customer_type|disclaimer|noise|unknown", "scope_names": ["..."], "fact_types": ["financial_metric|operational_metric|guidance|strategic_update|market_claim|risk_or_caveat"]}}]
}}

Slides:
{json.dumps(slide_digest, ensure_ascii=False)}
"""
    try:
        payload = _json_response(prompt)
    except Exception as exc:
        print(f"Inventory LLM failed; using fallback inventory: {type(exc).__name__}: {exc}")
        payload = {}
    if not payload.get("slide_plan"):
        payload["slide_plan"] = [
            {"slide_no": s["slide_no"], "title": s["title"], "primary_scope_level": "unknown", "scope_names": [], "fact_types": []}
            for s in slides
        ]
    payload.setdefault("document_type", "investor_presentation")
    payload.setdefault("period_label", PERIOD_LABEL)
    payload.setdefault("detected_segments", [])
    payload.setdefault("detected_sections", [])
    payload.setdefault("detected_metrics", [])
    return payload


def build_extraction_plan(inventory: dict, slides: list[dict]) -> dict:
    segment_names = [s.get("name") for s in inventory.get("detected_segments", []) if s.get("name")]
    actionable = []
    by_slide = {int(s["slide_no"]): s for s in slides}
    for item in inventory.get("slide_plan", []):
        slide_no = int(item.get("slide_no") or 0)
        if slide_no not in by_slide:
            continue
        level = str(item.get("primary_scope_level") or "unknown").lower()
        if level in {"disclaimer", "noise"}:
            continue
        actionable.append({
            "slide_no": slide_no,
            "title": item.get("title") or by_slide[slide_no]["title"],
            "scope_level": level,
            "scope_names": item.get("scope_names") or [],
            "fact_types": item.get("fact_types") or [],
        })
    return {
        "known_segments": segment_names,
        "actionable_slides": actionable,
        "rules": [
            "Extract only evidence-backed facts visible on the slide.",
            "Attach each fact to exactly one scope.",
            "Split multi-segment tables into one fact per segment/metric/value.",
            "Use unknown scope when evidence does not support the scope.",
        ],
    }


def extract_slide_facts(slide: dict, plan_item: dict, *, fact_catalog_text: str, known_segments: list[str]) -> list[dict]:
    prompt = f"""Extract presentation facts from one slide of an Indian listed-company investor presentation.

Company symbol: {SYMBOL}
Reporting period context: {PERIOD_LABEL}
Slide: {slide['slide_no']} - {slide['title']}
Inventory scope hint: {json.dumps(plan_item, ensure_ascii=False)}
Known segments from inventory: {known_segments}

Allowed fact_key values:
{fact_catalog_text}

Rules:
- Extract ONLY facts explicitly supported by this slide text/table/chart.
- Do not extract every number. Extract values that explain business performance, future growth, and management narrative.
- Allowed MVP layers: financial highlights, segment performance, business KPIs, order book/pipeline, guidance/outlook, capex/capacity, and demand/margin/outlook commentary.
- Use presentation_revenue, presentation_ebitda, presentation_pat, and presentation_ebitda_margin for company-level financial highlights shown in the deck.
- Every fact must have one scope object: level = company|segment|geography|product|channel|project|customer_type|unknown.
- If the slide has multiple segment columns/rows, create separate facts for each segment. Never merge segment values.
- Segment is a dimension. Use fact_key = segment_revenue with scope.level = segment, not a custom jewellery_revenue style key.
- If a number is company total, use scope.level = company and scope.name = null.
- If a segment is explicit, use scope.level = segment and the segment name exactly as shown or the canonical inventory name.
- Use value_lower/value_upper for guidance ranges and numeric_value as the midpoint. Preserve the exact range in evidence.
- Mark forward-looking statements as fact_type = guidance or strategic_update, not historical financial_metric.
- For commentary, include sentiment when clear: positive|neutral|mixed|negative.
- Include unit, period_label, source_page, evidence, and confidence.
- Do not infer missing numbers or totals.

Return JSON object:
{{"facts": [{{"fact_key": "...", "fact_type": "financial_metric|operational_metric|guidance|strategic_update|market_claim|risk_or_caveat", "numeric_value": 0.0, "value_lower": null, "value_upper": null, "value_text": null, "unit": "...", "period_label": "...", "period_end": "YYYY-MM-DD", "basis": "consolidated|standalone|not_applicable", "scope": {{"level": "segment", "name": "..."}}, "geography": null, "product": null, "channel": null, "project": null, "customer_type": null, "metric_context": null, "sentiment": null, "is_explicit_guidance": false, "source_page": 1, "evidence": "...", "confidence": 0.9}}]}}
If no facts found, return {{"facts": []}}.

Slide text:
{slide['text']}
"""
    try:
        payload = _json_response(prompt)
    except Exception as exc:
        print(f"LLM extraction skipped for slide {slide['slide_no']}: {type(exc).__name__}: {exc}")
        return []
    return [f for f in payload.get("facts") or [] if isinstance(f, dict)]


def extract_presentation_document(doc: dict, facts_catalog: dict, storage_to_fact: dict[str, str], fact_catalog_text: str) -> tuple[list[dict], dict]:
    rows: list[dict] = []
    slides = split_slides(doc["markdown"])
    inventory = build_inventory(doc, slides)
    extraction_plan = build_extraction_plan(inventory, slides)
    pack = {
        "event_id": doc.get("event_id", event_id),
        "document_id": doc["document_id"],
        "inventory": inventory,
        "extraction_plan": extraction_plan,
        "slides": slides,
    }
    print(
        f"{doc['document_type']}: inventory slides={len(slides)}, "
        f"segments={len(inventory.get('detected_segments', []))}, "
        f"actionable_slides={len(extraction_plan['actionable_slides'])}"
    )

    slide_by_no = {int(s["slide_no"]): s for s in slides}
    for plan_item in extraction_plan["actionable_slides"]:
        slide = slide_by_no.get(int(plan_item["slide_no"]))
        if not slide or not slide["text"].strip() or "*(empty page)*" in slide["text"].lower():
            continue
        for entry in extract_slide_facts(
            slide,
            plan_item,
            fact_catalog_text=fact_catalog_text,
            known_segments=extraction_plan["known_segments"],
        ):
            fk = entry.get("fact_key")
            canonical = storage_to_fact.get(str(fk), str(fk)) if fk else None
            if canonical not in facts_catalog:
                continue
            spec = facts_catalog[canonical]
            value_type = _fact_value_type(spec)
            value_lower = _optional_float(entry.get("value_lower") or entry.get("lower_value"))
            value_upper = _optional_float(entry.get("value_upper") or entry.get("upper_value"))
            numeric = None
            value_text = entry.get("value_text")
            if value_type == "numeric":
                raw_numeric = entry.get("numeric_value")
                if raw_numeric is None and value_lower is not None and value_upper is not None:
                    raw_numeric = (value_lower + value_upper) / 2.0
                try:
                    numeric = float(str(raw_numeric).replace(",", ""))
                except (TypeError, ValueError):
                    continue
            elif value_text is None:
                value_text = entry.get("evidence") or entry.get("source_text")

            scope = _scope_fields(entry)
            if canonical.startswith("presentation_") and scope["scope_level"] == "unknown":
                scope["scope_level"] = "company"
                scope["scope_name"] = None
            required_dimensions = set(spec.get("dimensions") or [])
            if "segment" in required_dimensions and not scope.get("segment"):
                continue

            source_page = entry.get("source_page") or entry.get("page") or slide["slide_no"]
            evidence = entry.get("evidence") or entry.get("source_text") or value_text or ""
            if not evidence:
                continue
            conf = max(0.0, min(1.0, float(entry.get("confidence") or 0.7)))
            rows.append({
                "event_id": doc.get("event_id", event_id),
                "document_id": doc["document_id"],
                "document_type": doc["document_type"],
                "fact_key": canonical,
                "fact_type": entry.get("fact_type") or str(spec.get("statement") or "presentation").lower(),
                "numeric_value": numeric,
                "value_text": value_text,
                "value_lower": value_lower,
                "value_upper": value_upper,
                "unit": _row_unit(spec, entry, value_type),
                "basis": (entry.get("basis") or "not_applicable").strip().lower(),
                "scope": "CURRENT",
                "period_end": entry.get("period_end") or PERIOD_END,
                "period_label": entry.get("period_label") or PERIOD_LABEL,
                "source_page": int(source_page) if str(source_page).isdigit() else slide["slide_no"],
                "evidence": evidence[:1500],
                "confidence": conf,
                "sentiment": entry.get("sentiment"),
                "is_explicit_guidance": _optional_bool(entry.get("is_explicit_guidance")),
                "extraction_method": "presentation_inventory_scoped_llm",
                **scope,
            })
    return rows, pack


all_observation_rows: list[dict] = []
presentation_inventories: list[dict] = []
for doc in parsed_documents:
    doc_type = doc["document_type"]
    facts_catalog = json.loads(Path(doc["catalog_path"]).read_text(encoding="utf-8"))
    storage_to_fact, fact_catalog_text = _catalog_index(facts_catalog)

    if doc_type == "investor_presentation":
        rows, pack = extract_presentation_document(doc, facts_catalog, storage_to_fact, fact_catalog_text)
        all_observation_rows.extend(rows)
        presentation_inventories.append(pack)
        continue

    deterministic_facts: list[dict] = []
    if doc_type == "financial_result":
        deterministic_facts = extract_scoped_facts_from_quarter_columns(
            doc["markdown"],
            target=reporting_period,
            fact_keys=set(facts_catalog.keys()),
            facts_catalog=facts_catalog,
        )
    det_by_scope_key = {
        (row.get("scope", "CURRENT"), row["fact_key"]): row
        for row in deterministic_facts
    }
    det_current_keys = {
        row["fact_key"] for row in deterministic_facts
        if row.get("scope", "CURRENT") == "CURRENT"
    }
    print(f"{doc_type}: deterministic table facts={len(deterministic_facts)}")

    raw_facts: list[dict] = list(deterministic_facts)
    missing_keys = set(facts_catalog.keys()) - det_current_keys
    for chunk in _chunk(doc["markdown"]):
        if not _llm_chunk_allowed(chunk, doc_type):
            continue
        for entry in extract_facts_from_chunk(chunk, doc_type=doc_type, fact_catalog_text=fact_catalog_text):
            fk = entry.get("fact_key")
            canonical = storage_to_fact.get(str(fk), str(fk)) if fk else None
            if canonical and ("CURRENT", canonical) in det_by_scope_key:
                continue
            if canonical and canonical not in missing_keys:
                continue
            raw_facts.append(entry)

    for entry in raw_facts:
        fk = entry.get("fact_key")
        if not fk:
            continue
        canonical = storage_to_fact.get(str(fk), str(fk))
        if canonical not in facts_catalog:
            continue
        spec = facts_catalog[canonical]
        value_type = _fact_value_type(spec)
        numeric = None
        value_text = entry.get("value_text")
        value_lower = _optional_float(entry.get("value_lower") or entry.get("lower_value"))
        value_upper = _optional_float(entry.get("value_upper") or entry.get("upper_value"))
        if value_type == "numeric":
            raw_numeric = entry.get("numeric_value")
            if raw_numeric is None and value_lower is not None and value_upper is not None:
                raw_numeric = (value_lower + value_upper) / 2.0
            if raw_numeric is None and spec.get("unit") == "boolean":
                raw_numeric = _bool_numeric(value_text)
            try:
                numeric = float(str(raw_numeric).replace(",", ""))
            except (TypeError, ValueError):
                continue
        elif value_text is None:
            value_text = entry.get("evidence") or entry.get("source_text")
        if canonical == "restatement_flag" and not _true_restatement_flag(entry):
            continue
        dimensions = _generic_dimension_fields(entry)
        required_dimensions = set(spec.get("dimensions") or [])
        if "segment" in required_dimensions and not dimensions.get("segment"):
            continue
        scope = str(entry.get("scope") or "CURRENT").upper()
        row_period_end = entry.get("period_end") or PERIOD_END
        row_period_label = entry.get("period_label") or (PERIOD_LABEL if scope == "CURRENT" else str(row_period_end))
        conf = float(entry.get("confidence") or 0.7)
        all_observation_rows.append({
            "event_id": doc.get("event_id", event_id),
            "document_id": doc["document_id"],
            "document_type": doc_type,
            "fact_key": canonical,
            "fact_type": entry.get("fact_type") or str(spec.get("statement") or "financial").lower(),
            "numeric_value": numeric,
            "value_text": value_text,
            "value_lower": value_lower,
            "value_upper": value_upper,
            "unit": _row_unit(spec, entry, value_type),
            "basis": (entry.get("basis") or "not_applicable").strip().lower(),
            **dimensions,
            "scope": scope,
            "period_end": row_period_end,
            "period_label": row_period_label,
            "extraction_method": entry.get("extraction_method") or "llm_chunk",
            "source_page": entry.get("source_page") or entry.get("page"),
            "evidence": entry.get("evidence") or entry.get("source_text") or value_text or "",
            "confidence": conf,
            "sentiment": entry.get("sentiment"),
            "is_explicit_guidance": _optional_bool(entry.get("is_explicit_guidance")),
        })

if not all_observation_rows:
    raise RuntimeError("No facts passed validation after extraction")

# Prefer consolidated numeric rows for legacy extracted_values only, but preserve
# text/commentary observations as separate evidence-backed rows.
preferred: dict[tuple, dict] = {}
text_rows: list[dict] = []
for row in all_observation_rows:
    if row["numeric_value"] is None:
        if row.get("value_text"):
            text_rows.append(row)
        continue
    key = (row["fact_key"], row.get("period_end") or PERIOD_END, *dimension_key(row), row.get("scope_level"), row.get("scope_name"))
    cur = preferred.get(key)
    if cur is None or (row["basis"] == "consolidated" and cur["basis"] != "consolidated"):
        preferred[key] = row
accepted_rows = list(preferred.values()) + text_rows

# ---- 4f. Persist observations, resolved facts, and legacy extracted values ----
def value_id(value_code: str, period_end: str, basis: str, *dims, source_page=None, source_text=None) -> str:
    dim = ":".join(str(v or "") for v in dims)
    extra = ":".join(str(v or "") for v in (source_page, source_text))
    return hashlib.sha256(
        f"{company_id}:{value_code}:{period_end}:{basis}:{dim}:{extra}".encode()
    ).hexdigest()


def observation_id_for(row: dict, event_id_value: str | None = None) -> str:
    event_id_value = event_id_value or row.get("event_id") or event_id
    dim = ":".join(str(v or "") for v in dimension_key(row))
    return hashlib.sha256(
        f"{company_id}:{event_id_value}:{row['document_id']}:{row['fact_key']}:{row['basis']}:{row.get('scope', 'CURRENT')}:{row.get('period_end') or PERIOD_END}:{row.get('scope_level') or ''}:{row.get('scope_name') or ''}:{dim}:{row.get('source_page') or ''}:{row.get('evidence', '')[:100]}".encode()
    ).hexdigest()


def resolved_fact_id_for(fact_code: str, *dims, scope_level=None, scope_name=None) -> str:
    dim = ":".join(str(v or "") for v in (*dims, scope_level, scope_name))
    return hashlib.sha256(f"{company_id}:{event_id}:{fact_code}:{dim}".encode()).hexdigest()


def _norm_source(value) -> str:
    return normalize_document_type(value)


def resolve_fact_observations(conn: sqlite3.Connection, fact_code: str, dims: tuple) -> dict | None:
    segment, geography, product, channel, project, customer_type, metric_context = dims
    observations = conn.execute(
        """
        SELECT fo.*, d.document_kind, fd.preferred_source
        FROM fact_observations fo
        LEFT JOIN documents d ON d.id = fo.document_id
        LEFT JOIN fact_definitions fd ON fd.fact_code = fo.fact_code
        WHERE fo.company_id = ? AND fo.event_id = ? AND fo.fact_code = ? AND fo.period = ?
          AND COALESCE(fo.segment, '') = COALESCE(?, '')
          AND COALESCE(fo.geography, '') = COALESCE(?, '')
          AND COALESCE(fo.product, '') = COALESCE(?, '')
          AND COALESCE(fo.channel, '') = COALESCE(?, '')
          AND COALESCE(fo.project, '') = COALESCE(?, '')
          AND COALESCE(fo.customer_type, '') = COALESCE(?, '')
          AND COALESCE(fo.metric_context, '') = COALESCE(?, '')
        """,
        (company_id, event_id, fact_code, PERIOD_LABEL, segment, geography, product, channel, project, customer_type, metric_context),
    ).fetchall()
    if not observations:
        return None
    preferred_source = _norm_source(observations[0]["preferred_source"])

    def score(obs):
        source_bonus = 1 if _norm_source(obs["document_kind"]) == preferred_source else 0
        return (source_bonus, float(obs["confidence"] or 0.0))

    selected = max(observations, key=score)
    values = [float(o["value"]) for o in observations if o["value"] is not None]
    if not values:
        status = "single_source_confirmed" if len(observations) == 1 else "text_fact_selected"
    elif len(values) == 1:
        status = "single_source_confirmed"
    elif max(values) - min(values) <= max(abs(float(selected["value"] or 0)) * 0.005, 1.0):
        status = "confirmed_with_rounding_difference"
    else:
        status = "conflict_needs_review"
    return {
        "resolved_fact_id": resolved_fact_id_for(fact_code, *dims, scope_level=selected["scope_level"], scope_name=selected["scope_name"]),
        "fact_code": fact_code,
        "resolved_value": selected["value"],
        "resolved_value_text": selected["value_text"],
        "unit": selected["unit"],
        "segment": selected["segment"],
        "geography": selected["geography"],
        "product": selected["product"],
        "channel": selected["channel"],
        "project": selected["project"],
        "customer_type": selected["customer_type"],
        "metric_context": selected["metric_context"],
        "scope_level": selected["scope_level"],
        "scope_name": selected["scope_name"],
        "fact_type": selected["fact_type"],
        "value_lower": selected["value_lower"],
        "value_upper": selected["value_upper"],
        "sentiment": selected["sentiment"],
        "is_explicit_guidance": selected["is_explicit_guidance"],
        "selected_observation_id": selected["observation_id"],
        "resolution_status": status,
        "confidence": float(selected["confidence"] or 0.0),
    }


def _insert_presentation_inventory(conn: sqlite3.Connection, pack: dict, event_id_value: str) -> None:
    inv_id = hashlib.sha256(f"{company_id}:{event_id_value}:{pack['document_id']}:inventory".encode()).hexdigest()
    conn.execute(
        """
        INSERT INTO presentation_document_inventory (
            id, company_id, event_id, document_id, period_label,
            inventory_json, extraction_plan_json
        ) VALUES (?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(id) DO UPDATE SET
            period_label = excluded.period_label,
            inventory_json = excluded.inventory_json,
            extraction_plan_json = excluded.extraction_plan_json
        """,
        (inv_id, company_id, event_id_value, pack["document_id"], PERIOD_LABEL,
         json.dumps(pack["inventory"], ensure_ascii=False),
         json.dumps(pack["extraction_plan"], ensure_ascii=False)),
    )
    for seg in pack["inventory"].get("detected_segments", []):
        name = seg.get("name")
        if not name:
            continue
        slug = _dimension_slug(name)
        sid = hashlib.sha256(f"{company_id}:{event_id_value}:{pack['document_id']}:{name}".encode()).hexdigest()
        conn.execute(
            """
            INSERT INTO presentation_segments (
                id, company_id, event_id, document_id, segment_name, segment_slug,
                aliases_json, slides_json, confidence
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(id) DO UPDATE SET
                segment_name = excluded.segment_name,
                segment_slug = excluded.segment_slug,
                aliases_json = excluded.aliases_json,
                slides_json = excluded.slides_json,
                confidence = excluded.confidence
            """,
            (sid, company_id, event_id_value, pack["document_id"], name, slug,
             json.dumps(seg.get("aliases") or [], ensure_ascii=False),
             json.dumps(seg.get("slides") or [], ensure_ascii=False),
             float(seg.get("confidence") or 0.0)),
        )


with db_connect() as conn:
    # Reruns are event-authoritative: clear stale extraction outputs for each
    # document event in this run, plus the primary aggregate event.
    event_ids_for_run = sorted({event_id} | {d.get("event_id", event_id) for d in parsed_documents})
    for run_event_id in event_ids_for_run:
        conn.execute("DELETE FROM intelligence_cards WHERE company_id = ? AND event_id = ?", (company_id, run_event_id))
        conn.execute("DELETE FROM signals WHERE company_id = ? AND event_id = ?", (company_id, run_event_id))
        conn.execute("DELETE FROM metrics WHERE company_id = ? AND event_id = ?", (company_id, run_event_id))
        conn.execute("DELETE FROM resolved_facts WHERE company_id = ? AND event_id = ?", (company_id, run_event_id))
        conn.execute("DELETE FROM fact_observations WHERE company_id = ? AND event_id = ?", (company_id, run_event_id))
        conn.execute("DELETE FROM extracted_values WHERE company_id = ? AND event_id = ?", (company_id, run_event_id))
        conn.execute("DELETE FROM presentation_document_inventory WHERE company_id = ? AND event_id = ?", (company_id, run_event_id))
        conn.execute("DELETE FROM presentation_segments WHERE company_id = ? AND event_id = ?", (company_id, run_event_id))

    for pack in presentation_inventories:
        pack_event_id = pack.get("event_id") or event_id
        for target_event_id in sorted({event_id, pack_event_id}):
            _insert_presentation_inventory(conn, pack, target_event_id)

    def persist_observation(row: dict, event_id_value: str) -> None:
        oid = observation_id_for(row, event_id_value)
        conn.execute(
            """
            INSERT INTO fact_observations (
                observation_id, company_id, event_id, document_id, fact_code,
                value, value_text, unit, period, source_page, source_text,
                segment, geography, product, channel, project, customer_type,
                metric_context, scope_level, scope_name, fact_type,
                extraction_method, value_lower, value_upper, sentiment,
                is_explicit_guidance, confidence
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(observation_id) DO UPDATE SET
                value = excluded.value,
                value_text = excluded.value_text,
                unit = excluded.unit,
                period = excluded.period,
                source_page = excluded.source_page,
                source_text = excluded.source_text,
                segment = excluded.segment,
                geography = excluded.geography,
                product = excluded.product,
                channel = excluded.channel,
                project = excluded.project,
                customer_type = excluded.customer_type,
                metric_context = excluded.metric_context,
                scope_level = excluded.scope_level,
                scope_name = excluded.scope_name,
                fact_type = excluded.fact_type,
                extraction_method = excluded.extraction_method,
                value_lower = excluded.value_lower,
                value_upper = excluded.value_upper,
                sentiment = excluded.sentiment,
                is_explicit_guidance = excluded.is_explicit_guidance,
                confidence = excluded.confidence
            """,
            (oid, company_id, event_id_value, row["document_id"], row["fact_key"], row["numeric_value"],
             row.get("value_text"), row["unit"], row.get("period_label") or PERIOD_LABEL,
             row.get("source_page"), row["evidence"], row.get("segment"), row.get("geography"),
             row.get("product"), row.get("channel"), row.get("project"), row.get("customer_type"),
             row.get("metric_context"), row.get("scope_level"), row.get("scope_name"), row.get("fact_type"),
             row.get("extraction_method"), row.get("value_lower"), row.get("value_upper"), row.get("sentiment"),
             row.get("is_explicit_guidance"), row["confidence"]),
        )

    for row in all_observation_rows:
        doc_event_id = row.get("event_id") or event_id
        persist_observation(row, doc_event_id)
        if doc_event_id != event_id:
            persist_observation(row, event_id)

    resolved_rows = []
    current_fact_keys = {
        (row["fact_key"], *dimension_key(row)) for row in all_observation_rows
        if row.get("scope", "CURRENT") == "CURRENT" and (row.get("period_end") or PERIOD_END) == PERIOD_END
    }
    for key in sorted(current_fact_keys, key=lambda k: tuple(str(v or "") for v in k)):
        fact_code, *dims = key
        resolved = resolve_fact_observations(conn, fact_code, tuple(dims))
        if resolved is None:
            continue
        resolved_rows.append(resolved)
        conn.execute(
            """
            INSERT INTO resolved_facts (
                resolved_fact_id, company_id, event_id, fact_code, resolved_value,
                resolved_value_text, unit, segment, geography, product, channel,
                project, customer_type, metric_context, scope_level, scope_name,
                fact_type, value_lower, value_upper, sentiment, is_explicit_guidance,
                selected_observation_id, resolution_status, confidence
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(resolved_fact_id) DO UPDATE SET
                resolved_value = excluded.resolved_value,
                resolved_value_text = excluded.resolved_value_text,
                unit = excluded.unit,
                segment = excluded.segment,
                geography = excluded.geography,
                product = excluded.product,
                channel = excluded.channel,
                project = excluded.project,
                customer_type = excluded.customer_type,
                metric_context = excluded.metric_context,
                scope_level = excluded.scope_level,
                scope_name = excluded.scope_name,
                fact_type = excluded.fact_type,
                value_lower = excluded.value_lower,
                value_upper = excluded.value_upper,
                sentiment = excluded.sentiment,
                is_explicit_guidance = excluded.is_explicit_guidance,
                selected_observation_id = excluded.selected_observation_id,
                resolution_status = excluded.resolution_status,
                confidence = excluded.confidence
            """,
            (resolved["resolved_fact_id"], company_id, event_id, resolved["fact_code"],
             resolved["resolved_value"], resolved.get("resolved_value_text"), resolved["unit"],
             resolved.get("segment"), resolved.get("geography"), resolved.get("product"),
             resolved.get("channel"), resolved.get("project"), resolved.get("customer_type"),
             resolved.get("metric_context"), resolved.get("scope_level"), resolved.get("scope_name"),
             resolved.get("fact_type"), resolved.get("value_lower"), resolved.get("value_upper"),
             resolved.get("sentiment"), resolved.get("is_explicit_guidance"),
             resolved["selected_observation_id"], resolved["resolution_status"], resolved["confidence"]),
        )

    for row in accepted_rows:
        row_period_end = row.get("period_end") or PERIOD_END
        text_identity = row.get("evidence", "")[:100] if row.get("numeric_value") is None else None
        vid = value_id(
            row["fact_key"], row_period_end, row["basis"], *dimension_key(row),
            row.get("scope_level"), row.get("scope_name"),
            source_page=row.get("source_page") if text_identity else None,
            source_text=text_identity,
        )
        conn.execute(
            """
            INSERT INTO extracted_values (
                id, company_id, event_id, value_code, value_numeric, value_text,
                unit, period_type, period_start, period_end, basis, segment,
                geography, product, channel, project, customer_type, metric_context,
                scope_level, scope_name, fact_type, value_lower, value_upper,
                sentiment, is_explicit_guidance, source_text, source_page, confidence
            ) VALUES (?, ?, ?, ?, ?, ?, ?, 'quarter', NULL, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(id) DO UPDATE SET
                event_id = excluded.event_id,
                value_numeric = excluded.value_numeric,
                value_text = excluded.value_text,
                unit = excluded.unit,
                period_end = excluded.period_end,
                basis = excluded.basis,
                segment = excluded.segment,
                geography = excluded.geography,
                product = excluded.product,
                channel = excluded.channel,
                project = excluded.project,
                customer_type = excluded.customer_type,
                metric_context = excluded.metric_context,
                scope_level = excluded.scope_level,
                scope_name = excluded.scope_name,
                fact_type = excluded.fact_type,
                value_lower = excluded.value_lower,
                value_upper = excluded.value_upper,
                sentiment = excluded.sentiment,
                is_explicit_guidance = excluded.is_explicit_guidance,
                source_text = excluded.source_text,
                source_page = excluded.source_page,
                confidence = excluded.confidence
            """,
            (vid, company_id, event_id, row["fact_key"], row["numeric_value"], row.get("value_text"),
             row["unit"], row_period_end, row["basis"], row.get("segment"), row.get("geography"),
             row.get("product"), row.get("channel"), row.get("project"), row.get("customer_type"),
             row.get("metric_context"), row.get("scope_level"), row.get("scope_name"), row.get("fact_type"),
             row.get("value_lower"), row.get("value_upper"), row.get("sentiment"),
             row.get("is_explicit_guidance"), row["evidence"], row.get("source_page"), row["confidence"]),
        )
    for doc in parsed_documents:
        conn.execute(
            "UPDATE events SET status = 'processed', fiscal_year = ?, fiscal_quarter = ?, document_id = ? WHERE id = ?",
            (PERIOD_FY_START, PERIOD_QUARTER, doc["document_id"], doc.get("event_id", event_id)),
        )
    conn.execute(
        "UPDATE documents SET status = 'processed' WHERE id IN (%s)" % ",".join("?" for _ in parsed_documents),
        tuple(d["document_id"] for d in parsed_documents),
    )
    conn.commit()
    resolved_facts_current = {
        (r["fact_code"], r.get("segment"), r.get("geography"), r.get("product"), r.get("channel"),
         r.get("project"), r.get("customer_type"), r.get("metric_context")): r
        for r in resolved_rows
    }

print(f"\nObserved {len(all_observation_rows)} fact(s) across {len(parsed_documents)} document(s) for {PERIOD_LABEL}")
for row in sorted([r for r in accepted_rows if r.get("numeric_value") is not None], key=lambda r: (r.get("period_end") or "", r["fact_key"], dimension_sort_key(r))):
    scope_note = row.get("scope", "CURRENT")
    fact_label = f"{row['fact_key']}{_dimension_suffix(row)}"
    print(f"  {scope_note:<7} {fact_label:<54} {row['numeric_value']:>14,.2f} {row['unit'] or '':<14} [{row['document_type']}]")

context_rows = [
    row for row in all_observation_rows
    if row.get("value_text") and row.get("numeric_value") is None
]
if context_rows:
    print("\nContext observations:")
    for row in sorted(context_rows, key=lambda r: (r.get("period_end") or "", r["fact_key"], dimension_sort_key(r), str(r.get("source_page") or ""))):
        scope_note = row.get("scope", "CURRENT")
        fact_label = f"{row['fact_key']}{_dimension_suffix(row)}"
        snippet = str(row.get("value_text") or "").strip().replace("\n", " ")[:140]
        print(f"  {scope_note:<7} {fact_label:<54} {snippet} [{row['document_type']}]")

if presentation_inventories:
    detected_segment_names = sorted({
        seg.get("name")
        for pack in presentation_inventories
        for seg in pack["inventory"].get("detected_segments", [])
        if seg.get("name")
    })
    print(f"\nInvestor presentation inventory: {len(detected_segment_names)} detected segment(s)")
    for name in detected_segment_names[:20]:
        print(f"  - {name}")
print(f"Resolved {len(resolved_facts_current)} trusted fact(s) for metrics/signals")


## STEP 5 / 7  -  METRICS

In [ ]:
# =============================================================================
# STEP 5 / 7  -  METRICS
# Load the metric catalog, load CURRENT / prior-year / prior-quarter facts,
# evaluate the catalog formulas, and persist computed rows into `metrics`.
# (YoY / QoQ metrics only compute once those prior quarters exist in the DB.)
# =============================================================================
from statistics import mean

metrics_catalog = json.loads((CATALOG_DIR / "metrics.json").read_text(encoding="utf-8"))

# ---- 5a. Load fact pools for CURRENT, prior-year, prior-quarter --------------
def prior_year_end() -> str:
    return quarter_end_date(PERIOD_QUARTER, PERIOD_FY_START - 1).isoformat()


def prior_quarter_end() -> str:
    if PERIOD_QUARTER == 1:
        q, fy = 4, PERIOD_FY_START - 1
    else:
        q, fy = PERIOD_QUARTER - 1, PERIOD_FY_START
    return quarter_end_date(q, fy).isoformat()


def event_id_for_period(fiscal_year: int, fiscal_quarter: int) -> str | None:
    with db_connect() as conn:
        row = conn.execute(
            """
            SELECT id FROM events
            WHERE company_id = ? AND fiscal_year = ? AND fiscal_quarter = ?
            ORDER BY CASE LOWER(COALESCE(event_type, ''))
                WHEN 'quarterly_result' THEN 0
                WHEN 'quarterly result' THEN 0
                WHEN 'financial results' THEN 0
                WHEN 'financial_result' THEN 0
                ELSE 1
            END, event_date DESC, id DESC
            LIMIT 1
            """,
            (company_id, fiscal_year, fiscal_quarter),
        ).fetchone()
    return row["id"] if row else None


def load_facts(event_id_value: str | None, period_end: str | None = None) -> dict[str, dict]:
    pool: dict[str, dict] = {}
    with db_connect() as conn:
        if event_id_value:
            rows = conn.execute(
                """
                SELECT resolved_fact_id, fact_code, resolved_value, unit, resolution_status, confidence,
                       segment, geography, metric_context
                FROM resolved_facts
                WHERE company_id = ? AND event_id = ?
                  AND COALESCE(segment, '') = ''
                  AND COALESCE(geography, '') = ''
                  AND COALESCE(product, '') = ''
                  AND COALESCE(channel, '') = ''
                  AND COALESCE(project, '') = ''
                  AND COALESCE(customer_type, '') = ''
                  AND COALESCE(metric_context, '') = ''
                """,
                (company_id, event_id_value),
            ).fetchall()
            for r in rows:
                if r["resolved_value"] is None or r["resolution_status"] == "conflict_needs_review":
                    continue
                pool.setdefault(r["fact_code"], {
                    "value": float(r["resolved_value"]),
                    "unit": r["unit"],
                    "resolved_fact_id": r["resolved_fact_id"],
                    "confidence": r["confidence"],
                })
        rows = []
        if period_end is not None:
            rows = conn.execute(
                """
                SELECT value_code, value_numeric, unit FROM extracted_values
                WHERE company_id = ? AND period_end = ?
                  AND COALESCE(segment, '') = ''
                  AND COALESCE(geography, '') = ''
                  AND COALESCE(product, '') = ''
                  AND COALESCE(channel, '') = ''
                  AND COALESCE(project, '') = ''
                  AND COALESCE(customer_type, '') = ''
                  AND COALESCE(metric_context, '') = ''
                ORDER BY CASE basis WHEN 'consolidated' THEN 0 ELSE 1 END
                """,
                (company_id, period_end),
            ).fetchall()
    for r in rows:
        if r["value_numeric"] is None:
            continue
        pool.setdefault(r["value_code"], {
            "value": float(r["value_numeric"]),
            "unit": r["unit"],
            "resolved_fact_id": value_id(r["value_code"], period_end, "legacy"),
            "confidence": None,
        })
    return pool


py_event_id = event_id_for_period(PERIOD_FY_START - 1, PERIOD_QUARTER)
if PERIOD_QUARTER == 1:
    pq_event_id = event_id_for_period(PERIOD_FY_START - 1, 4)
else:
    pq_event_id = event_id_for_period(PERIOD_FY_START, PERIOD_QUARTER - 1)

facts_current = load_facts(event_id, PERIOD_END)
facts_py = load_facts(py_event_id, prior_year_end())
facts_pq = load_facts(pq_event_id, prior_quarter_end())
SCOPE_POOLS = {"CURRENT": facts_current, "PY": facts_py, "PQ": facts_pq}

# ---- 5b. Unit -> crore-equivalent scaling for cross-scope comparison ---------
_UNIT_SCALE = {
    "crore": 1.0, "crores": 1.0, "cr": 1.0, "lakh": 0.01, "lakhs": 0.01,
    "lac": 0.01, "lacs": 0.01, "million": 0.1, "mn": 0.1, "billion": 100.0,
    "bn": 100.0, "thousand": 1e-5,
}
_PASS_THROUGH = {"%", "percent", "pct", "bps", "rs", "rs.", "inr", "x", "times", "days"}


def crore_scale(unit):
    if not unit:
        return None
    u = str(unit).strip().lower()
    if u in _PASS_THROUGH:
        return None
    return _UNIT_SCALE.get(u)


RUNTIME_PARAMS = {"annualization_factor": 4.0 / PERIOD_QUARTER}


def _fact_scopes(inputs: list[dict]) -> set[str]:
    return {
        inp.get("scope", "CURRENT").upper()
        for inp in inputs
        if "fact_key" in inp
    }


def resolve_inputs(
    inputs: list[dict],
    *,
    eval_scope: str = "CURRENT",
    metrics_by_scope: dict[str, dict[str, float]] | None = None,
):
    metrics_by_scope = metrics_by_scope or {}
    use_comparable = len(_fact_scopes(inputs)) > 1
    resolved: dict[str, float] = {}
    for inp in inputs:
        if "runtime_parameter" in inp:
            val = RUNTIME_PARAMS.get(inp["runtime_parameter"])
            if val is None:
                return None
            resolved[inp["var"]] = val
            continue

        if "metric_key" in inp:
            ref_scope = inp.get("scope", "CURRENT").upper()
            val = metrics_by_scope.get(ref_scope, {}).get(inp["metric_key"])
            if val is None:
                return None
            resolved[inp["var"]] = val
            continue

        if "fact_key" not in inp:
            return None

        abs_scope = inp.get("scope", "CURRENT").upper()
        pool_scope = eval_scope.upper() if abs_scope == "CURRENT" else abs_scope
        detail = SCOPE_POOLS.get(pool_scope, {}).get(inp["fact_key"])
        if detail is None:
            if inp.get("optional"):
                resolved[inp["var"]] = 0.0
                continue
            return None
        val = detail["value"]
        if use_comparable:
            scale = crore_scale(detail["unit"])
            if scale is not None:
                val = val * scale
        resolved[inp["var"]] = val
    return resolved


def safe_eval(formula: str, variables: dict[str, float]):
    namespace = {
        "__builtins__": {},
        "min": min,
        "max": max,
        "abs": abs,
        "average": lambda a, b: (a + b) / 2,
        **variables,
    }
    try:
        result = eval(formula, namespace)  # trusted catalog formulas only
    except (ZeroDivisionError, TypeError, NameError):
        return None
    return float(result) if isinstance(result, (int, float)) else None


def input_labels(inputs: list[dict]) -> list[str]:
    labels: list[str] = []
    for inp in inputs:
        if "fact_key" in inp:
            labels.append(inp["fact_key"])
        elif "metric_key" in inp:
            labels.append(inp["metric_key"])
    return labels


def input_fact_ids(inputs: list[dict], eval_scope: str = "CURRENT") -> list[str]:
    ids: list[str] = []
    for inp in inputs:
        if "fact_key" not in inp:
            continue
        abs_scope = inp.get("scope", "CURRENT").upper()
        pool_scope = eval_scope.upper() if abs_scope == "CURRENT" else abs_scope
        detail = SCOPE_POOLS.get(pool_scope, {}).get(inp["fact_key"])
        if detail and detail.get("resolved_fact_id"):
            ids.append(detail["resolved_fact_id"])
    return sorted(set(ids))


# ---- 5c. Compute and persist metrics -----------------------------------------
_EVAL_SCOPES = ("CURRENT", "PY", "PQ")
metrics_by_scope: dict[str, dict[str, float]] = {s: {} for s in _EVAL_SCOPES}

while True:
    progress = False
    for code, spec in metrics_catalog.items():
        inputs_spec = spec.get("inputs") or []
        for eval_scope in _EVAL_SCOPES:
            if code in metrics_by_scope[eval_scope]:
                continue
            variables = resolve_inputs(
                inputs_spec,
                eval_scope=eval_scope,
                metrics_by_scope=metrics_by_scope,
            )
            if variables is None:
                continue
            value = safe_eval(spec["formula"], variables)
            if value is None:
                continue
            metrics_by_scope[eval_scope][code] = round(value, 2)
            progress = True
    if not progress:
        break

computed_metrics: list[dict] = []
for code, spec in metrics_catalog.items():
    value = metrics_by_scope["CURRENT"].get(code)
    if value is None:
        continue
    computed_metrics.append({
        "metric_id": hashlib.sha256(
            f"{company_id}:{event_id}:{code}".encode()
        ).hexdigest(),
        "metric_key": code,
        "name": spec.get("name", code),
        "value": value,
        "unit": spec.get("unit"),
        "category": spec.get("category"),
        "formula": spec["formula"],
        "inputs": sorted(set(input_labels(spec.get("inputs") or []))),
        "input_fact_ids": input_fact_ids(spec.get("inputs") or []),
    })


# Presentation coverage/quality metrics are additive to the normal financial
# result metrics. They describe extraction shape instead of audited performance.
presentation_rows = [
    row for row in globals().get("all_observation_rows", [])
    if row.get("document_type") == "investor_presentation"
    and row.get("scope", "CURRENT") == "CURRENT"
    and (row.get("period_end") or PERIOD_END) == PERIOD_END
]
if presentation_rows:
    scope_counts: dict[str, int] = {}
    segment_counts: dict[str, int] = {}
    fact_type_counts: dict[str, int] = {}
    for row in presentation_rows:
        scope_level = row.get("scope_level") or "unknown"
        scope_counts[scope_level] = scope_counts.get(scope_level, 0) + 1
        if row.get("segment"):
            segment_counts[row["segment"]] = segment_counts.get(row["segment"], 0) + 1
        fact_type = row.get("fact_type") or "unknown"
        fact_type_counts[fact_type] = fact_type_counts.get(fact_type, 0) + 1

    inventory_segment_names = {
        seg.get("name")
        for pack in globals().get("presentation_inventories", [])
        for seg in pack.get("inventory", {}).get("detected_segments", [])
        if seg.get("name")
    }
    guidance_fact_keys = {"revenue_growth_guidance", "margin_guidance", "management_outlook", "segment_outlook"}
    guidance_count = sum(
        1 for row in presentation_rows
        if row.get("fact_type") == "guidance" or row.get("fact_key") in guidance_fact_keys
    )
    confidence_values = [float(r.get("confidence") or 0.0) for r in presentation_rows]

    def _presentation_metric(metric_key: str, name: str, value: float, unit: str, category: str, formula: str) -> dict:
        return {
            "metric_id": hashlib.sha256(f"{company_id}:{event_id}:{metric_key}".encode()).hexdigest(),
            "metric_key": metric_key,
            "name": name,
            "value": round(float(value), 4),
            "unit": unit,
            "category": category,
            "formula": formula,
            "inputs": [],
            "input_fact_ids": [],
        }

    computed_metrics.extend([
        _presentation_metric("presentation_fact_count", "Presentation Fact Count", len(presentation_rows), "count", "coverage", "count(scoped_presentation_facts)"),
        _presentation_metric("presentation_segment_count", "Detected Segment Count", len(inventory_segment_names) or len(segment_counts), "count", "coverage", "count(distinct segment)"),
        _presentation_metric("presentation_segment_fact_count", "Segment Scoped Fact Count", scope_counts.get("segment", 0), "count", "coverage", "count(facts where scope_level='segment')"),
        _presentation_metric("presentation_company_fact_count", "Company Scoped Fact Count", scope_counts.get("company", 0), "count", "coverage", "count(facts where scope_level='company')"),
        _presentation_metric("presentation_unknown_scope_fact_count", "Unknown Scope Fact Count", scope_counts.get("unknown", 0), "count", "quality", "count(facts where scope_level='unknown')"),
        _presentation_metric("presentation_average_confidence", "Average Presentation Extraction Confidence", mean(confidence_values) if confidence_values else 0.0, "score", "quality", "avg(fact.confidence)"),
        _presentation_metric("presentation_guidance_fact_count", "Guidance Fact Count", guidance_count, "count", "content", "count(guidance facts)"),
    ])

with db_connect() as conn:
    conn.execute(
        "DELETE FROM metrics WHERE company_id = ? AND event_id = ?",
        (company_id, event_id),
    )
    for m in computed_metrics:
        conn.execute(
            """
            INSERT INTO metrics (
                metric_id, company_id, event_id, metric_code, value,
                unit, input_fact_ids, formula
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (m["metric_id"], company_id, event_id, m["metric_key"], m["value"], m["unit"],
             ", ".join(m["input_fact_ids"]), m["formula"]),
        )
    conn.commit()

print(f"Computed {len(computed_metrics)} metric(s) for {PERIOD_LABEL}")
if not facts_py:
    print("  (no prior-year facts in DB -> YoY metrics skipped)")
if not facts_pq:
    print("  (no prior-quarter facts in DB -> QoQ metrics skipped)")
print()
for m in computed_metrics:
    print(f"  {m['name']:<32} {m['value']:>12,.2f} {m['unit'] or '':<5}  [{m['category']}]")

## STEP 6 / 7  -  SIGNALS

In [ ]:
# =============================================================================
# STEP 6 / 7  -  SIGNALS
# Evaluate the signal-rule catalog against the computed metrics and persist
# every fired signal into the `signals` table.
# =============================================================================
signals_catalog = json.loads((CATALOG_DIR / "signals.json").read_text(encoding="utf-8"))

metrics_by_key = {m["metric_key"]: m for m in computed_metrics}
facts_by_key = dict(facts_current)
for row in globals().get("accepted_rows", []):
    if "fact_key" not in row:
        continue
    if row.get("scope", "CURRENT") != "CURRENT" or (row.get("period_end") or PERIOD_END) != PERIOD_END:
        continue
    # Signal rules are scalar today; dimensioned facts stay queryable in storage
    # but should not overwrite scalar facts by bare fact_key.
    if any(row.get(k) for k in ("segment", "geography", "product", "channel", "project", "customer_type", "metric_context")):
        continue
    value = row.get("value", row.get("numeric_value", row.get("value_text")))
    detail = facts_by_key.setdefault(row["fact_key"], {})
    detail.update({"value": value, "unit": row.get("unit")})
    if "resolved_fact_id" not in detail and "resolved_facts_current" in globals():
        rf = resolved_facts_current.get((row["fact_key"], None, None, None, None, None, None, None))
        if rf:
            detail["resolved_fact_id"] = rf["resolved_fact_id"]

with db_connect() as conn:
    resolved_fact_ids_by_code = {
        r["fact_code"]: r["resolved_fact_id"]
        for r in conn.execute(
            """
            SELECT fact_code, resolved_fact_id FROM resolved_facts
            WHERE company_id = ? AND event_id = ?
              AND COALESCE(segment, '') = ''
              AND COALESCE(geography, '') = ''
              AND COALESCE(product, '') = ''
              AND COALESCE(channel, '') = ''
              AND COALESCE(project, '') = ''
              AND COALESCE(customer_type, '') = ''
              AND COALESCE(metric_context, '') = ''
            """,
            (company_id, event_id),
        ).fetchall()
    }


def resolved_fact_id_for(fact_code: str) -> str:
    detail = facts_by_key.get(fact_code) or {}
    if detail.get("resolved_fact_id"):
        return detail["resolved_fact_id"]
    if fact_code in resolved_fact_ids_by_code:
        return resolved_fact_ids_by_code[fact_code]
    return hashlib.sha256(f"{company_id}:{event_id}:{fact_code}".encode()).hexdigest()


_OPS = {
    ">": lambda a, b: a > b,
    ">=": lambda a, b: a >= b,
    "<": lambda a, b: a < b,
    "<=": lambda a, b: a <= b,
    "==": lambda a, b: a == b,
    "!=": lambda a, b: a != b,
    "IN": lambda a, b: isinstance(b, (list, tuple, set)) and a in b,
    "NOT IN": lambda a, b: isinstance(b, (list, tuple, set)) and a not in b,
    "ABS_GT": lambda a, b: abs(a) > b,
}


def eval_leaf(rule: dict) -> bool:
    if "metric_key" in rule:
        ref = metrics_by_key.get(rule["metric_key"])
    elif "fact_key" in rule:
        ref = facts_by_key.get(rule["fact_key"])
    else:
        raise ValueError(f"Malformed rule leaf: {rule}")
    if ref is None:
        return False
    op = rule["op"].upper() if isinstance(rule["op"], str) else rule["op"]
    if op not in _OPS:
        raise ValueError(f"Unsupported rule operator: {rule['op']}")
    return _OPS[op](ref["value"], rule["value"])


def eval_rule(rule: dict) -> bool:
    if "all" in rule:
        return all(eval_rule(child) for child in rule["all"])
    if "any" in rule:
        return any(eval_rule(child) for child in rule["any"])
    if "metric_key" in rule or "fact_key" in rule:
        return eval_leaf(rule)
    raise ValueError(f"Malformed rule: {rule}")


def rule_metric_keys(rule: dict) -> list[str]:
    if "metric_key" in rule:
        return [rule["metric_key"]]
    keys: list[str] = []
    for k in ("all", "any"):
        for child in rule.get(k, []):
            keys.extend(rule_metric_keys(child))
    return keys


def rule_fact_keys(rule: dict) -> list[str]:
    if "fact_key" in rule:
        return [rule["fact_key"]]
    keys: list[str] = []
    for k in ("all", "any"):
        for child in rule.get(k, []):
            keys.extend(rule_fact_keys(child))
    return keys


def format_rule(rule: dict) -> str:
    if "metric_key" in rule:
        return f"{rule['metric_key']} {rule['op']} {rule['value']}"
    if "fact_key" in rule:
        return f"{rule['fact_key']} {rule['op']} {rule['value']}"
    if "all" in rule:
        return " AND ".join(f"({format_rule(c)})" for c in rule["all"])
    if "any" in rule:
        return " OR ".join(f"({format_rule(c)})" for c in rule["any"])
    return ""


fired_signals: list[dict] = []
for code, spec in signals_catalog.items():
    rule = spec.get("rule")
    if not rule:
        continue
    # A rule can only fire if every metric/fact it references is available.
    keys = rule_metric_keys(rule)
    fact_keys = rule_fact_keys(rule)
    if not all(k in metrics_by_key for k in keys):
        continue
    if not all(k in facts_by_key for k in fact_keys):
        continue
    if not eval_rule(rule):
        continue
    trigger = {k: metrics_by_key[k]["value"] for k in keys if k in metrics_by_key}
    trigger.update({k: facts_by_key[k]["value"] for k in fact_keys if k in facts_by_key})
    supporting_metric_ids = [metrics_by_key[k]["metric_id"] for k in keys if k in metrics_by_key]
    supporting_fact_ids = set(resolved_fact_id_for(k) for k in fact_keys if k in facts_by_key)
    for k in keys:
        supporting_fact_ids.update(metrics_by_key.get(k, {}).get("input_fact_ids") or [])
    fired_signals.append({
        "signal_key": code,
        "title": spec.get("name", code),
        "description": spec.get("description", ""),
        "direction": spec.get("direction"),
        "severity": spec.get("severity"),
        "category": spec.get("category"),
        "metric_keys": keys,
        "fact_keys": fact_keys,
        "supporting_metric_ids": supporting_metric_ids,
        "supporting_fact_ids": sorted(supporting_fact_ids),
        "trigger_values": trigger,
        "rule_text": format_rule(rule),
    })


# Presentation-specific coverage/quality signals.
presentation_signals_catalog = {
    "multi_segment_presentation": {
        "name": "Multi-Segment Presentation",
        "category": "coverage",
        "direction": "NEUTRAL",
        "severity": "LOW",
        "description": "Presentation contains more than one detected segment.",
    },
    "segment_facts_available": {
        "name": "Segment Facts Available",
        "category": "coverage",
        "direction": "POSITIVE",
        "severity": "MEDIUM",
        "description": "Segment-scoped facts were extracted separately from company facts.",
    },
    "guidance_present": {
        "name": "Guidance Present",
        "category": "content",
        "direction": "NEUTRAL",
        "severity": "MEDIUM",
        "description": "Forward-looking or guidance facts were found in the presentation.",
    },
    "unknown_scope_needs_review": {
        "name": "Unknown Scope Needs Review",
        "category": "quality",
        "direction": "NEGATIVE",
        "severity": "HIGH",
        "description": "Some presentation facts could not be tied to company, segment, geography, product, channel, project, or customer type scope.",
    },
    "low_extraction_confidence": {
        "name": "Low Presentation Extraction Confidence",
        "category": "quality",
        "direction": "NEGATIVE",
        "severity": "HIGH",
        "description": "Average presentation extraction confidence is below the review threshold.",
    },
}

if "presentation_fact_count" in metrics_by_key:
    def presentation_metric_value(key: str) -> float:
        return float(metrics_by_key.get(key, {}).get("value") or 0.0)

    presentation_checks = [
        ("multi_segment_presentation", presentation_metric_value("presentation_segment_count") >= 2, ["presentation_segment_count"]),
        ("segment_facts_available", presentation_metric_value("presentation_segment_fact_count") > 0, ["presentation_segment_fact_count"]),
        ("guidance_present", presentation_metric_value("presentation_guidance_fact_count") > 0, ["presentation_guidance_fact_count"]),
        ("unknown_scope_needs_review", presentation_metric_value("presentation_unknown_scope_fact_count") > 0, ["presentation_unknown_scope_fact_count"]),
        ("low_extraction_confidence", presentation_metric_value("presentation_average_confidence") < 0.70, ["presentation_average_confidence"]),
    ]

    for signal_key, fired, metric_keys in presentation_checks:
        if not fired:
            continue
        spec = presentation_signals_catalog[signal_key]
        fired_signals.append({
            "signal_key": signal_key,
            "title": spec["name"],
            "description": spec["description"],
            "direction": spec["direction"],
            "severity": spec["severity"],
            "category": spec["category"],
            "metric_keys": metric_keys,
            "fact_keys": [],
            "supporting_metric_ids": [metrics_by_key[k]["metric_id"] for k in metric_keys if k in metrics_by_key],
            "supporting_fact_ids": [],
            "trigger_values": {k: metrics_by_key[k]["value"] for k in metric_keys if k in metrics_by_key},
            "rule_text": ", ".join(metric_keys),
        })

# ---- Persist fired signals ---------------------------------------------------
with db_connect() as conn:
    conn.execute(
        "DELETE FROM signals WHERE company_id = ? AND event_id = ?",
        (company_id, event_id),
    )
    for s in fired_signals:
        sid = hashlib.sha256(
            f"{company_id}:{event_id}:{s['signal_key']}".encode()
        ).hexdigest()
        conn.execute(
            """
            INSERT INTO signals (
                signal_id, company_id, event_id, signal_code, severity,
                direction, supporting_metric_ids, supporting_fact_ids
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (sid, company_id, event_id, s["signal_key"], s["severity"], s["direction"],
             ", ".join(s["supporting_metric_ids"]),
             ", ".join(s["supporting_fact_ids"])),
        )
    conn.commit()

print(f"Evaluated {len(signals_catalog)} signal rules -> {len(fired_signals)} fired\n")
for s in fired_signals:
    print(f"  [{s['severity']}/{s['direction']}] {s['title']}")
    print(f"        rule: {s['rule_text']}  ->  {s['trigger_values']}")

## STEP 7 / 7  -  ALERTS

In [ ]:
# =============================================================================
# STEP 7 / 7  -  ALERTS
# Read the fired signals back from the DB and present them as alerts,
# then print a final per-event DB summary.
# =============================================================================
alert_signals_catalog = dict(globals().get("signals_catalog") or json.loads(
    (CATALOG_DIR / "signals.json").read_text(encoding="utf-8")
))
alert_signals_catalog.update(globals().get("presentation_signals_catalog", {}))


def card_confidence(severity: str | None) -> str:
    sev = str(severity or "").upper()
    if sev in {"CRITICAL", "HIGH"}:
        return "high"
    if sev == "MEDIUM":
        return "medium"
    return "low"


with db_connect() as conn:
    alert_rows = conn.execute(
        """
        SELECT signal_id, signal_code, severity, direction, supporting_metric_ids, supporting_fact_ids
        FROM signals WHERE company_id = ? AND event_id = ?
        """,
        (company_id, event_id),
    ).fetchall()

    conn.execute(
        "DELETE FROM intelligence_cards WHERE company_id = ? AND event_id = ?",
        (company_id, event_id),
    )
    for r in alert_rows:
        spec = alert_signals_catalog.get(r["signal_code"], {})
        card_title = spec.get("name", r["signal_code"])
        card_id = hashlib.sha256(
            f"{company_id}:{event_id}:{r['signal_id']}:card".encode()
        ).hexdigest()
        conn.execute(
            """
            INSERT INTO intelligence_cards (
                card_id, company_id, event_id, card_title, signal_id,
                confidence, display_status
            ) VALUES (?, ?, ?, ?, ?, ?, 'published')
            """,
            (card_id, company_id, event_id, card_title, r["signal_id"],
             card_confidence(r["severity"])),
        )
    conn.commit()

    counts = {
        "fact_observations": conn.execute(
            "SELECT COUNT(*) c FROM fact_observations WHERE event_id = ?", (event_id,)
        ).fetchone()["c"],
        "resolved_facts": conn.execute(
            "SELECT COUNT(*) c FROM resolved_facts WHERE event_id = ?", (event_id,)
        ).fetchone()["c"],
        "extracted_values": conn.execute(
            "SELECT COUNT(*) c FROM extracted_values WHERE event_id = ?", (event_id,)
        ).fetchone()["c"],
        "metrics": conn.execute(
            "SELECT COUNT(*) c FROM metrics WHERE event_id = ?", (event_id,)
        ).fetchone()["c"],
        "signals": conn.execute(
            "SELECT COUNT(*) c FROM signals WHERE event_id = ?", (event_id,)
        ).fetchone()["c"],
        "intelligence_cards": conn.execute(
            "SELECT COUNT(*) c FROM intelligence_cards WHERE event_id = ?", (event_id,)
        ).fetchone()["c"],
        "presentation_segments": conn.execute(
            "SELECT COUNT(*) c FROM presentation_segments WHERE event_id = ?", (event_id,)
        ).fetchone()["c"],
        "presentation_inventories": conn.execute(
            "SELECT COUNT(*) c FROM presentation_document_inventory WHERE event_id = ?", (event_id,)
        ).fetchone()["c"],
    }
    scope_rows = conn.execute(
        """
        SELECT COALESCE(scope_level, 'unscoped') scope_level, COUNT(*) c
        FROM fact_observations
        WHERE event_id = ? AND document_id IN (
            SELECT id FROM documents WHERE company_id = ? AND LOWER(COALESCE(document_kind, '')) IN ('investor_presentation', 'investor presentation')
        )
        GROUP BY COALESCE(scope_level, 'unscoped')
        ORDER BY c DESC, scope_level
        """,
        (event_id, company_id),
    ).fetchall()
    segment_rows = conn.execute(
        """
        SELECT segment, COUNT(*) c
        FROM fact_observations
        WHERE event_id = ? AND segment IS NOT NULL
        GROUP BY segment
        ORDER BY c DESC, segment
        """,
        (event_id,),
    ).fetchall()

_SEVERITY_ORDER = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3}

print("=" * 70)
print(f"INTELLIGENCE CARDS  -  {SYMBOL}  -  {EVENT_TYPE}  -  {PERIOD_LABEL}")
print("=" * 70)

if not alert_rows:
    print("\nNo signals triggered for this filing.")
else:
    ordered = sorted(alert_rows, key=lambda r: _SEVERITY_ORDER.get(str(r["severity"]).upper(), 9))
    for r in ordered:
        spec = alert_signals_catalog.get(r["signal_code"], {})
        title = spec.get("name", r["signal_code"])
        print(f"\n  [{r['severity']}] {title}  ({r['direction']})")
        if spec.get("description"):
            print(f"      {spec['description']}")
        if r["supporting_metric_ids"]:
            print(f"      metrics: {r['supporting_metric_ids']}")
        if r["supporting_fact_ids"]:
            print(f"      facts  : {r['supporting_fact_ids']}")

if counts["presentation_inventories"] or counts["presentation_segments"] or scope_rows:
    print("\nPresentation extraction summary:")
    print(f"  inventories       : {counts['presentation_inventories']}")
    print(f"  detected segments : {counts['presentation_segments']}")
    if scope_rows:
        print("  scope counts:")
        for r in scope_rows:
            print(f"    {r['scope_level']:<16} {r['c']:>4}")
    if segment_rows:
        print("  segments with facts:")
        for r in segment_rows[:20]:
            print(f"    {r['segment']:<24} {r['c']:>4}")

print("\n" + "-" * 70)
print(f"DB summary for event {event_id[:12]}...:")
print(f"  fact_observations    : {counts['fact_observations']}")
print(f"  resolved_facts       : {counts['resolved_facts']}")
print(f"  extracted_values     : {counts['extracted_values']}")
print(f"  metrics              : {counts['metrics']}")
print(f"  signals              : {counts['signals']}")
print(f"  cards                : {counts['intelligence_cards']}")
print(f"  presentation_segments: {counts['presentation_segments']}")
print(f"\nAll seven steps complete. Data persisted to {DB_PATH}")
